# PBL6 - Notebook 2/2: HUẤN LUYỆN + ĐÁNH GIÁ (FGI Deepfake Detection)

**Cần có trước:** file dữ liệu đã tiền xử lý trên Drive: `MyDrive/pbl6_work.tar` (hoặc `MyDrive/pbl6/dfdc_processed.tar` do notebook 1 tạo ra). Notebook tự tìm file đầu tiên có mặt.

**Trước khi chạy:**
1. Runtime → Change runtime type → **GPU**.
2. Chạy lần lượt từ trên xuống.

**Nếu Colab bị ngắt:** chạy lại từ đầu notebook. Cell giải nén sẽ khôi phục dữ liệu, cell khởi tạo model sẽ tự **resume từ checkpoint mới nhất** trong `MyDrive/pbl6/models_<RUN_TAG>`.

**Checkpoint/log** lưu ở `MyDrive/pbl6/models_<RUN_TAG>` và `MyDrive/pbl6/logs_<RUN_TAG>`.

**Bản v5_mouth:** dữ liệu là ảnh **crop vùng miệng** (tạo bởi `pbl6_1_preprocess_mouth.ipynb`, lưu ở `MyDrive/pbl6_mouth/`), để mô hình tập trung vào độ lệch khẩu hình – âm thanh thay vì cả khuôn mặt. Cell suy luận ở cuối dùng đúng cách crop này. Crop miệng dùng YuNet + landmark môi của MediaPipe (`CROP_MODE = 'mouth_lm'`); cell suy luận cần `mediapipe` và sẽ báo lỗi (không âm thầm đổi cách crop) nếu thiếu.

In [ ]:
!nvidia-smi | head -15

In [ ]:
!pip install -q tensorboardX scipy scikit-learn opencv-python-headless tqdm

In [ ]:
# Mount Google Drive (checkpoint/log + file du lieu da tien xu ly)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/pbl6', exist_ok=True)

## 1. Khôi phục dữ liệu đã tiền xử lý từ Drive

Giải nén file `.tar` **trực tiếp từ Drive** ra ổ cục bộ `/mnt/local-scratch/work` (không copy tar về trước, nên chỉ cần dung lượng bằng 1 lần kích thước dữ liệu thay vì 2 lần). Đọc từng file nhỏ từ Drive rất chậm nên dữ liệu phải nằm ở ổ cục bộ khi train. Nếu `/mnt/local-scratch/work` đã có dữ liệu (cùng phiên Colab) thì bỏ qua.

> **Lưu ý (dữ liệu crop miệng):** nếu `/mnt/local-scratch/work` còn dữ liệu của lần chạy cũ (crop mặt hoặc chưa crop) thì cell dưới sẽ bỏ qua bước giải nén và dùng lại dữ liệu cũ. Hãy dùng runtime mới hoặc xóa thư mục đó trước. Cũng nên xóa checkpoint cũ trong `DRIVE_DIR` để không tiếp tục train từ model của dữ liệu cũ.

In [ ]:
import os, shutil, subprocess

WORK_DIR = '/mnt/local-scratch/work'
# File du lieu tren Drive (lay file dau tien tim thay)
TAR_CANDIDATES = [
    '/content/drive/MyDrive/pbl6_mouth/dfdc_processed.tar',   # ban crop MIENG (notebook 1 moi)
    '/content/drive/MyDrive/pbl6_10_12/dfdc_processed.tar',   # do notebook 1 (ban crop mat) tao ra
    '/content/drive/MyDrive/pbl6/dfdc_processed.tar',
    '/content/drive/MyDrive/pbl6_work.tar',
    '/content/drive/MyDrive/pbl6/pbl6_work.tar',
]
TAR_DRIVE = next((p for p in TAR_CANDIDATES if os.path.exists(p)), None)
TAR_LOCAL = '/content/data_work.tar'
os.makedirs(WORK_DIR, exist_ok=True)

if os.path.exists(os.path.join(WORK_DIR, 'train_split.csv')) and os.path.isdir(os.path.join(WORK_DIR, 'train', 'pytmp')):
    print('Du lieu da co san o /mnt/local-scratch/work, bo qua buoc giai nen.')
else:
    assert TAR_DRIVE is not None, 'Khong thay file tar tren Drive. Da tim: ' + ', '.join(TAR_CANDIDATES)

    # Don file tar cuc bo do lan chay truoc bi loi giua chung (neu co) de giai phong cho
    if os.path.exists(TAR_LOCAL):
        os.remove(TAR_LOCAL)

    tar_size = os.path.getsize(TAR_DRIVE)
    free = shutil.disk_usage('/mnt/local-scratch').free
    print('Dung file:', TAR_DRIVE)
    print(f'  Kich thuoc tar  : {tar_size/1024**3:.2f} GB')
    print(f'  O dia con trong : {free/1024**3:.2f} GB')
    assert tar_size > 0, 'File tar rong - co the Drive chua dong bo xong, doi roi chay lai.'
    assert free > tar_size * 1.05, (
        'Khong du cho trong tren o dia Colab de giai nen. Cach xu ly: '
        '(1) chon runtime co o dia lon hon (xem o dia o tab Tai nguyen), '
        '(2) giam du lieu: dat MAX_REAL/MAX_FAKE nho hon o notebook 1 roi nen lai, '
        '(3) Runtime -> Disconnect and delete runtime de don o dia roi chay lai.')

    head = subprocess.run(f'tar -tf "{TAR_DRIVE}" | head -8', shell=True, capture_output=True, text=True)
    print('Vai muc dau trong tar:')
    print(head.stdout)

    # Giai nen TRUC TIEP tu Drive (doc tuan tu 1 file lon): khong copy tar ve o cuc bo nen chi can ~1x dung luong, khong phai 2x
    print('Dang giai nen truc tiep tu Drive (co the mat vai phut den vai chuc phut)...')
    r = subprocess.run(['tar', '-xf', TAR_DRIVE, '-C', WORK_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print('CANH BAO: tar bao loi (file .tar tren Drive co the chua tai du, bi cat cut, hoac het cho trong):')
        print(r.stderr[-1500:])
    assert r.returncode == 0, 'tar loi - xem thong bao phia tren, dung lai'
    print('Giai nen xong.')

print('Noi dung /mnt/local-scratch/work:', os.listdir(WORK_DIR))

## 2. Import thư viện

In [ ]:
import os
import sys
import copy
import glob
import time
import math
import random
import numbers
import pickle
import shutil
import argparse
import re
import csv
import subprocess
import collections.abc as collections

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from collections import deque
from datetime import datetime

from PIL import ImageOps, Image
from scipy.io import wavfile
from scipy.interpolate import interp1d
from scipy import signal
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, log_loss
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils import data
import torch.utils.data
import torchvision
from torchvision import transforms
import torchvision.transforms.functional as TF
from tensorboardX import SummaryWriter
from tqdm import tqdm

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Cấu hình

In [ ]:
class Config:
    # Du lieu da tien xu ly (giai nen tu dfdc_processed.tar)
    WORK_DIR = '/mnt/local-scratch/work'   # o dia local cua Colab (nhanh)
    # Checkpoint + log luu tren Google Drive de khong mat khi Colab ngat ket noi
    DRIVE_DIR = '/content/drive/MyDrive/pbl6'
    TRAIN_DIR = os.path.join(WORK_DIR, 'train')
    TEST_DIR = os.path.join(WORK_DIR, 'test')

    # Training hyperparameters
    BATCH_SIZE = 32
    NUM_WORKERS = 10   # da doi tu 2 -> 10 (Colab thuong chi co ~2 CPU: neu bi cham/het RAM thi giam lai)
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    EPOCHS = 30
    IMG_DIM = 224
    SPATIAL_SIZE = 28
    NET = 'resnet18'

    # Model options
    WITH_ATTENTION = True
    RESIDUAL_CONN = False
    USING_PSEUDO_FAKE = True

    # Pseudo-fake augmentation parameters
    AUD_MIN_FAKE_LEN = 2
    AUD_MAX_FAKE_LEN = -1
    VIS_MIN_FAKE_LEN = 2
    VIS_MAX_FAKE_LEN = -1

    # Xac suat ap pseudo-fake cho moi mau train (truoc day co dinh 0.5).
    # Pseudo-fake luon gan nhan fake, nen neu train goc 50/50 thi ti le real trong train = 0.5 * (1 - PSEUDO_FAKE_PROB):
    # 0.5 -> 25% real (lech nang), 0.25 -> 37.5% real. Muc 0.25 la diem khoi dau, co the chinh.
    PSEUDO_FAKE_PROB = 0.25

    # Augmentation cho train (lat ngang + doi sang/tuong phan/bao hoa nhe, cung tham so cho ca 30 frame).
    # Ap dung SAU khi tron pseudo-fake; valid/test khong dung. Dat False de tat.
    USE_AUGMENTATION = True

    # Augmentation AM THANH (chong "chen nhac / tieng on -> bi coi la fake"): tron nhac tong hop / noise vao audio.
    # Ap dung cho CA real lan fake (nhan giu nguyen) de model khong hoc tat "co nhac = fake".
    AUDIO_AUG_PROB = 0.4          # xac suat ap dung cho moi mau train (0 = tat)
    MUSIC_SNR_DB = (0, 20)        # giong/nhac: cang thap cang nhac to (0 dB = nhac to ngang giong)
    NOISE_SNR_DB = (5, 25)
    # Moi epoch do them valid_music_auc: tap valid (co dinh theo index) bi chen nhac/noise 100%
    EVAL_MUSIC_VALID = True

    # LR scheduler: ReduceLROnPlateau theo SELECT SCORE (cung diem voi chon best, xem ben duoi)
    LR_FACTOR = 0.5
    LR_PATIENCE = 1      # giam LR sau (LR_PATIENCE + 1) lan validate ma score khong tang
    MIN_LR = 1e-6

    # DataLoader: so batch nap truoc moi worker (chi dung khi NUM_WORKERS > 0)
    PREFETCH_FACTOR = 4

    # Misc
    PRINT_FREQ = 10
    SEED = 0
    SAVE_ALL = False

    # Gioi han so VIDEO GOC (tong train+valid+test) moi lop; None = lay het
    MAX_REAL = 13000
    MAX_FAKE = 13000

    # Train / Valid / Test split ratios (must sum to < 1.0; remainder -> test)
    TRAIN_RATIO = 0.85
    VALID_RATIO = 0.1   # remaining 0.15 becomes the held-out test split

    # Run validation every N epochs (1 = every epoch)
    VALID_FREQ = 1

    # ---- Chon BEST MODEL tren tap valid (muc video) ----
    # score = base - phat
    #   base = sum(w_k * thanh_phan_k) / sum(w_k)    (chi cac key co trong SELECT_WEIGHTS)
    #   gap  = AUC_train-sach - AUC_valid
    #   phat = GAP_PENALTY * max(0, gap - GAP_TOL)
    # Thanh phan ho tro: 'auc', 'f1' (= F1 o nguong toi uu quet 0.05..0.95 tren valid),
    #                    'acc' (accuracy valid o nguong 0.5), 'loss' (= 1 - min(logloss, 1)),
    #                    'music_auc' (AUC valid khi bi chen nhac/noise; can EVAL_MUSIC_VALID=True).
    # Key la -> bao loi (khong am tham bo qua). Muon them acc / loss: them vao dict, vd 'acc': 0.20, 'loss': 0.20
    SELECT_WEIGHTS = {'auc': 0.40, 'f1': 0.30, 'loss': 0.30}   # AUC + F1(nguong toi uu) + logloss valid; co the them 'acc', 'music_auc'
    GAP_PENALTY = 1.0
    GAP_TOL = 0.03
    # Epoch chi duoc xet lam best khi gap <= MAX_GAP_AUC (hop le) va score > best_score + SCORE_MIN_DELTA
    MAX_GAP_AUC = 0.10
    SCORE_MIN_DELTA = 0.002
    # Tap train "sach" co dinh: so video MOI LOP (tat pseudo-fake + augmentation) de do AUC_train-sach
    CLEAN_TRAIN_PER_CLASS = 200
    # Dung som: sau N epoch lien tiep khong co best moi (None = tat)
    EARLY_STOP_PATIENCE = 8
    # Dung khi overfit (gap > MAX_GAP_AUC) lien tiep N epoch va khong co best moi (None = tat)
    OVERFIT_STOP_PATIENCE = 3
    # Doi RUN_TAG moi khi doi cong thuc/cau hinh de khong resume checkpoint cu (checkpoint/log luu o models_<TAG>, logs_<TAG>)
    RUN_TAG = 'v5_mouth'

cfg = Config()
print('Configuration loaded.')
print(f'  Working Dir: {cfg.WORK_DIR}')
print(f'  Epochs: {cfg.EPOCHS}, BS: {cfg.BATCH_SIZE}, LR: {cfg.LEARNING_RATE}')
print(f'  Attention: {cfg.WITH_ATTENTION}, Pseudo-Fake: {cfg.USING_PSEUDO_FAKE}')
print(f'  NUM_WORKERS: {cfg.NUM_WORKERS}, Early stop patience: {cfg.EARLY_STOP_PATIENCE}, Overfit stop: {cfg.OVERFIT_STOP_PATIENCE}')
print(f'  Select: weights={cfg.SELECT_WEIGHTS}, gap_penalty={cfg.GAP_PENALTY}, gap_tol={cfg.GAP_TOL}, '
      f'max_gap={cfg.MAX_GAP_AUC}, min_delta={cfg.SCORE_MIN_DELTA}, RUN_TAG={cfg.RUN_TAG}')
print(f'  Pseudo-fake prob: {cfg.PSEUDO_FAKE_PROB}, Augmentation: {cfg.USE_AUGMENTATION}, '
      f'LR scheduler: x{cfg.LR_FACTOR} sau {cfg.LR_PATIENCE + 1} epoch khong tang, prefetch: {cfg.PREFETCH_FACTOR}')
print(f'  Max videos -> real: {cfg.MAX_REAL}, fake: {cfg.MAX_FAKE}')
print(f'  Split ratios -> train: {cfg.TRAIN_RATIO}, valid: {cfg.VALID_RATIO}, '
      f'test: {round(1 - cfg.TRAIN_RATIO - cfg.VALID_RATIO, 3)}')

## 3b. Kiểm tra dữ liệu và tự sửa nếu thiếu file split

Cell này đảm bảo có `train/pytmp` và 3 file CSV. Nếu CSV bị thiếu (hoặc nằm lệch thư mục) thì tự tạo lại từ `train/pytmp` bằng đúng cách chia của notebook 1 (chia theo video, `sorted` + cùng `SEED`).

In [ ]:
import csv, random, glob, shutil

CSV_NAMES = ['train_split.csv', 'valid_split.csv', 'test_imbalance_split.csv']

# 1) Dua train/pytmp ve dung cho neu giai nen bi long trong thu muc con
pytmp_target = os.path.join(cfg.WORK_DIR, 'train', 'pytmp')
if not os.path.isdir(pytmp_target):
    hits = [p for p in glob.glob(os.path.join(cfg.WORK_DIR, '**', 'pytmp'), recursive=True) if os.path.isdir(p)]
    if hits:
        os.makedirs(os.path.dirname(pytmp_target), exist_ok=True)
        shutil.move(hits[0], pytmp_target)
        print('Da dua pytmp ve:', pytmp_target)
assert os.path.isdir(pytmp_target), ('Khong thay thu muc train/pytmp. File dfdc_processed.tar co the bi hong/thieu. '
                                     'Hay chay lai notebook 1 (cell nen va luu Drive) roi xoa /mnt/local-scratch/work.')

# 2) Dua CSV ve dung cho neu bi long
for name in CSV_NAMES:
    dst = os.path.join(cfg.WORK_DIR, name)
    if not os.path.exists(dst):
        hits = glob.glob(os.path.join(cfg.WORK_DIR, '**', name), recursive=True)
        if hits:
            shutil.move(hits[0], dst)
            print('Da dua', name, 've dung cho')

# 2b) Chuan hoa duong dan trong CSV (CSV tao tren Windows co dau \\ hoac duong dan tuyet doi nhu G:\\...)
def _fix_csv(path):
    with open(path, newline='') as f:
        rows = [r for r in csv.reader(f) if r]
    changed = False
    out = []
    for row in rows:
        new = list(row)
        for i in (0, 1):
            if i < len(new):
                c = new[i].replace('\\', '/')
                k = c.find('train/pytmp/')
                if k > 0:
                    c = c[k:]
                if c != new[i]:
                    changed = True
                new[i] = c
        out.append(new)
    if changed:
        with open(path, 'w', newline='') as f:
            csv.writer(f).writerows(out)
        print('Da chuan hoa duong dan trong', os.path.basename(path))

for name in CSV_NAMES:
    p = os.path.join(cfg.WORK_DIR, name)
    if os.path.exists(p):
        _fix_csv(p)
        with open(p, newline='') as f:
            first = next(csv.reader(f), None)
        if first:
            assert os.path.isdir(os.path.join(cfg.WORK_DIR, first[0])), (
                f'{name}: duong dan {first[0]} khong ton tai trong {cfg.WORK_DIR}. '
                'Kiem tra cau truc ben trong file tar.')

missing = [n for n in CSV_NAMES if not os.path.exists(os.path.join(cfg.WORK_DIR, n))]

# 3) Ham chia (giong notebook 1)
def write_csv_list(data_list, path):
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f, delimiter=',')
        for row in data_list:
            if row:
                writer.writerow(row)
    print(f'Split saved to {path} ({len(data_list)} entries)')


def generate_splits(work_dir, train_ratio=0.7, valid_ratio=0.15, max_real=None, max_fake=None):
    """
    Split preprocessed videos into train / valid / test at the VIDEO level
    (so chunks from the same source video never leak across splits), then
    write out the chunk-level rows for each split to CSV.

    train_ratio + valid_ratio must leave room for a test split
    (test_ratio = 1 - train_ratio - valid_ratio).
    """
    assert 0 < train_ratio < 1, 'train_ratio must be between 0 and 1'
    assert 0 < valid_ratio < 1, 'valid_ratio must be between 0 and 1'
    assert train_ratio + valid_ratio < 1, 'train_ratio + valid_ratio must leave room for a test split'

    pytmp_real = os.path.join(work_dir, 'train', 'pytmp', 'real')
    pytmp_fake = os.path.join(work_dir, 'train', 'pytmp', 'fake')

    real_videos = []
    fake_videos = []

    if os.path.exists(pytmp_real):
        real_videos = [d for d in sorted(os.listdir(pytmp_real))
                       if os.path.isdir(os.path.join(pytmp_real, d))]
    if os.path.exists(pytmp_fake):
        fake_videos = [d for d in sorted(os.listdir(pytmp_fake))
                       if os.path.isdir(os.path.join(pytmp_fake, d))]

    print(f'Found {len(real_videos)} real videos, {len(fake_videos)} fake videos')

    random.shuffle(real_videos)
    random.shuffle(fake_videos)

    # Gioi han so video goc moi lop (sau khi shuffle nen lay ngau nhien, co dinh theo SEED)
    if max_real is not None:
        real_videos = real_videos[:max_real]
    if max_fake is not None:
        fake_videos = fake_videos[:max_fake]
    print(f'Sau khi gioi han: {len(real_videos)} real, {len(fake_videos)} fake')

    def three_way_split(videos, train_ratio, valid_ratio):
        n = len(videos)
        n_train = int(n * train_ratio)
        n_valid = int(n * valid_ratio)
        train = videos[:n_train]
        valid = videos[n_train:n_train + n_valid]
        test = videos[n_train + n_valid:]
        return train, valid, test

    real_train, real_valid, real_test = three_way_split(real_videos, train_ratio, valid_ratio)
    fake_train, fake_valid, fake_test = three_way_split(fake_videos, train_ratio, valid_ratio)

    print(f'Train: {len(real_train)} real, {len(fake_train)} fake')
    print(f'Valid: {len(real_valid)} real, {len(fake_valid)} fake')
    print(f'Test : {len(real_test)} real, {len(fake_test)} fake')

    base_train = os.path.join('train', 'pytmp')

    def collect_rows(video_names, pytmp_dir, label):
        rows = []
        for video_name in video_names:
            video_dir = os.path.join(pytmp_dir, video_name)
            if not os.path.isdir(video_dir):
                continue
            for chunk in sorted(os.listdir(video_dir)):
                chunk_path = os.path.join(video_dir, chunk)
                if os.path.isdir(chunk_path):
                    audio_file = chunk + '.wav'
                    audio_path = os.path.join(video_dir, audio_file)
                    if os.path.exists(audio_path):
                        rows.append([
                            os.path.join(base_train, label, video_name, chunk),
                            os.path.join(base_train, label, video_name, audio_file),
                            label
                        ])
        return rows

    train_set = collect_rows(fake_train, pytmp_fake, 'fake') + collect_rows(real_train, pytmp_real, 'real')
    valid_set = collect_rows(fake_valid, pytmp_fake, 'fake') + collect_rows(real_valid, pytmp_real, 'real')
    test_set = collect_rows(fake_test, pytmp_fake, 'fake') + collect_rows(real_test, pytmp_real, 'real')

    random.shuffle(train_set)
    random.shuffle(valid_set)
    random.shuffle(test_set)

    write_csv_list(train_set, os.path.join(work_dir, 'train_split.csv'))
    write_csv_list(valid_set, os.path.join(work_dir, 'valid_split.csv'))
    write_csv_list(test_set, os.path.join(work_dir, 'test_imbalance_split.csv'))

    return len(train_set), len(valid_set), len(test_set)

if missing or cfg.MAX_REAL is not None or cfg.MAX_FAKE is not None:
    print('Tao lai split tu train/pytmp (thieu CSV hoac co gioi han MAX_REAL/MAX_FAKE; chia theo video, sorted + SEED nen ket qua co dinh)')
    random.seed(cfg.SEED)
    n_train, n_valid, n_test = generate_splits(cfg.WORK_DIR, train_ratio=cfg.TRAIN_RATIO, valid_ratio=cfg.VALID_RATIO,
                                               max_real=cfg.MAX_REAL, max_fake=cfg.MAX_FAKE)
    print(f'Train samples: {n_train}, Valid samples: {n_valid}, Test samples: {n_test}')
else:
    print('Du 3 file split:', CSV_NAMES)

In [ ]:
# ====================== KIEM TRA CAN BANG LOP SAU KHI CHIA SPLIT ======================
import pandas as pd

def check_split_balance(csv_path, split_name):
    df = pd.read_csv(csv_path, header=None, names=['video_path', 'audio_path', 'label'])
    counts = df['label'].value_counts()
    n_real = counts.get('real', 0)
    n_fake = counts.get('fake', 0)
    total = n_real + n_fake
    n_unique_videos = df['video_path'].apply(lambda p: p.split('/')[-2]).nunique()
    print(f'{split_name:>6} | Real: {n_real:5d} ({n_real/total*100:5.1f}%)  '
          f'Fake: {n_fake:5d} ({n_fake/total*100:5.1f}%)  '
          f'Tong chunk: {total:5d}  |  So video goc: {n_unique_videos}')

print('Ti le real/fake tren tung tap sau khi chia (theo chunk, khong phai theo video):')
print('-' * 90)
check_split_balance(os.path.join(cfg.WORK_DIR, 'train_split.csv'), 'Train')
check_split_balance(os.path.join(cfg.WORK_DIR, 'valid_split.csv'), 'Valid')
check_split_balance(os.path.join(cfg.WORK_DIR, 'test_imbalance_split.csv'), 'Test')

## 4. Hàm tiện ích

In [ ]:
def min_max_normalize(value, min_value, max_value):
    return (value - min_value) / (max_value - min_value + 0.00000001)


def save_checkpoint(state, is_best=0, gap=1, filename='models/checkpoint.pth.tar', keep_all=False):
    torch.save(state, filename)
    last_epoch_path = os.path.join(os.path.dirname(filename),
                                   'epoch%s.pth.tar' % str(state['epoch'] - gap))
    if (state['epoch'] - gap) == 50:
        os.makedirs(os.path.join(os.path.dirname(filename), 'halfepochs_results'), exist_ok=True)
        if os.path.exists(last_epoch_path):
            os.rename(last_epoch_path,
                      os.path.join(os.path.dirname(filename), 'halfepochs_results',
                                   'epoch%s.pth.tar' % str(state['epoch'] - gap)))
        past_best = glob.glob(os.path.join(os.path.dirname(filename), 'model_best_*.pth.tar'))
        for i in past_best:
            os.rename(i, os.path.join(os.path.dirname(filename), 'halfepochs_results', os.path.basename(i)))
    if not keep_all:
        try:
            os.remove(last_epoch_path)
        except:
            pass
    if is_best:
        past_best = glob.glob(os.path.join(os.path.dirname(filename), 'model_best_*.pth.tar'))
        for i in past_best:
            try:
                os.remove(i)
            except:
                pass
        torch.save(state, os.path.join(os.path.dirname(filename),
                                       'model_best_epoch%s.pth.tar' % str(state['epoch'])))


def write_log(content, epoch, filename):
    if not os.path.exists(filename):
        log_file = open(filename, 'w')
    else:
        log_file = open(filename, 'a')
    log_file.write('## Epoch %d:\n' % epoch)
    log_file.write('time: %s\n' % str(datetime.now()))
    log_file.write(content + '\n\n')
    log_file.close()


def denorm(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]):
    assert len(mean) == len(std) == 3
    inv_mean = [-mean[i] / std[i] for i in range(3)]
    inv_std = [1 / i for i in std]
    return transforms.Normalize(mean=inv_mean, std=inv_std)


class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.local_history = deque([])
        self.local_avg = 0
        self.history = []
        self.dict = {}
        self.save_dict = {}

    def update(self, val, n=1, history=0, step=5):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
        if history:
            self.history.append(val)
        if step > 0:
            self.local_history.append(val)
            if len(self.local_history) > step:
                self.local_history.popleft()
            self.local_avg = np.average(self.local_history)

    def dict_update(self, val, key):
        if key in self.dict.keys():
            self.dict[key].append(val)
        else:
            self.dict[key] = [val]

    def __len__(self):
        return self.count

print('Utility functions defined.')

## 5. Augmentation

In [ ]:
class Padding:
    def __init__(self, pad):
        self.pad = pad
    def __call__(self, img):
        return ImageOps.expand(img, border=self.pad, fill=0)


class Scale:
    def __init__(self, size, interpolation=Image.NEAREST):
        assert isinstance(size, int) or (isinstance(size, collections.Iterable) and len(size) == 2)
        self.size = size
        self.interpolation = interpolation

    def __call__(self, imgmap):
        img1 = imgmap[0]
        if isinstance(self.size, int):
            w, h = img1.size
            if (w <= h and w == self.size) or (h <= w and h == self.size):
                return imgmap
            if w < h:
                ow = self.size
                oh = int(self.size * h / w)
                return [i.resize((ow, oh), self.interpolation) for i in imgmap]
            else:
                oh = self.size
                ow = int(self.size * w / h)
                return [i.resize((ow, oh), self.interpolation) for i in imgmap]
        else:
            return [i.resize(self.size, self.interpolation) for i in imgmap]


class CenterCrop:
    def __init__(self, size, consistent=True):
        if isinstance(size, numbers.Number):
            self.size = (int(size), int(size))
        else:
            self.size = size
    def __call__(self, imgmap):
        img1 = imgmap[0]
        w, h = img1.size
        th, tw = self.size
        x1 = int(round((w - tw) / 2.))
        y1 = int(round((h - th) / 2.))
        return [i.crop((x1, y1, x1 + tw, y1 + th)) for i in imgmap]


class RandomHorizontalFlip:
    def __init__(self, consistent=True, command=None):
        self.consistent = consistent
        if command == 'left':
            self.threshold = 0
        elif command == 'right':
            self.threshold = 1
        else:
            self.threshold = 0.5
    def __call__(self, imgmap):
        if self.consistent:
            if random.random() < self.threshold:
                return [i.transpose(Image.FLIP_LEFT_RIGHT) for i in imgmap]
            else:
                return imgmap
        else:
            result = []
            for i in imgmap:
                if random.random() < self.threshold:
                    result.append(i.transpose(Image.FLIP_LEFT_RIGHT))
                else:
                    result.append(i)
            return result


class ToTensor:
    def __call__(self, imgmap):
        totensor = transforms.ToTensor()
        return [totensor(i) for i in imgmap]


class Normalize:
    def __init__(self, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
        self.mean = mean
        self.std = std
    def __call__(self, imgmap):
        normalize = transforms.Normalize(mean=self.mean, std=self.std)
        return [normalize(i) for i in imgmap]

print('Augmentation transforms defined.')

## 6. Backbone ResNet 2D3D

In [ ]:
def conv3x3x3(in_planes, out_planes, stride=1, bias=False):
    return nn.Conv3d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=bias)

def conv1x3x3(in_planes, out_planes, stride=1, bias=False):
    return nn.Conv3d(in_planes, out_planes, kernel_size=(1, 3, 3),
                     stride=(1, stride, stride), padding=(0, 1, 1), bias=bias)


def downsample_basic_block(x, planes, stride):
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = torch.Tensor(
        out.size(0), planes - out.size(1), out.size(2), out.size(3),
        out.size(4)).zero_()
    if isinstance(out.data, torch.cuda.FloatTensor):
        zero_pads = zero_pads.cuda()
    out = Variable(torch.cat([out.data, zero_pads], dim=1))
    return out


class BasicBlock3d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 track_running_stats=True, use_final_relu=True):
        super(BasicBlock3d, self).__init__()
        bias = False
        self.use_final_relu = use_final_relu
        self.conv1 = conv3x3x3(inplanes, planes, stride, bias=bias)
        self.bn1 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3x3(planes, planes, bias=bias)
        self.bn2 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        if self.use_final_relu:
            out = self.relu(out)
        return out


class BasicBlock2d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 track_running_stats=True, use_final_relu=True):
        super(BasicBlock2d, self).__init__()
        bias = False
        self.use_final_relu = use_final_relu
        self.conv1 = conv1x3x3(inplanes, planes, stride, bias=bias)
        self.bn1 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv1x3x3(planes, planes, bias=bias)
        self.bn2 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        if self.use_final_relu:
            out = self.relu(out)
        return out


class ResNet2d3d_half(nn.Module):
    def __init__(self, block, layers, track_running_stats=True):
        super(ResNet2d3d_half, self).__init__()
        self.inplanes = 64
        self.track_running_stats = track_running_stats
        bias = False
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2),
                               padding=(0, 3, 3), bias=bias)
        self.bn1 = nn.BatchNorm3d(64, track_running_stats=track_running_stats)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))

        if not isinstance(block, list):
            block = [block] * 4

        self.layer1 = self._make_layer(block[0], 64, layers[0])
        self.layer2 = self._make_layer(block[1], 128, layers[1], stride=2, is_final=True)

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                m.weight = nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1, is_final=False):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if block == BasicBlock2d:
                customized_stride = (1, stride, stride)
            else:
                customized_stride = stride
            downsample = nn.Sequential(
                nn.Conv3d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=customized_stride, bias=False),
                nn.BatchNorm3d(planes * block.expansion,
                               track_running_stats=self.track_running_stats)
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample,
                            track_running_stats=self.track_running_stats))
        self.inplanes = planes * block.expansion
        if is_final:
            for i in range(1, blocks - 1):
                layers.append(block(self.inplanes, planes,
                                    track_running_stats=self.track_running_stats))
            layers.append(block(self.inplanes, planes,
                                track_running_stats=self.track_running_stats,
                                use_final_relu=False))
        else:
            for i in range(1, blocks):
                layers.append(block(self.inplanes, planes,
                                    track_running_stats=self.track_running_stats))
        return nn.Sequential(*layers)

    def forward(self, x, return_intermediate=False):
        x0 = self.conv1(x)
        x0 = self.bn1(x0)
        x0 = self.relu(x0)
        x0 = self.maxpool(x0)
        x1 = self.layer1(x0)
        x = self.layer2(x1)
        if return_intermediate:
            return x, x1
        else:
            return x


def resnet18_2d3d_half(**kwargs):
    model = ResNet2d3d_half([BasicBlock3d, BasicBlock3d], [2, 2], **kwargs)
    return model


def resnet34_2d3d_half(**kwargs):
    model = ResNet2d3d_half([BasicBlock3d, BasicBlock3d], [6, 3], **kwargs)
    return model


def select_resnet_half(network, track_running_stats=True):
    param = {'feature_size': 512}
    if network == 'resnet18':
        model = resnet18_2d3d_half(track_running_stats=track_running_stats)
        param['feature_size'] = 128
    elif network == 'resnet34':
        model = resnet34_2d3d_half(track_running_stats=track_running_stats)
        param['feature_size'] = 128
    else:
        raise IOError('model type is wrong')
    return model, param


def neq_load_customized(model, pretrained_dict):
    model_dict = model.state_dict()
    tmp = {}
    print('\n=======Check Weights Loading======')
    print('Weights not used from pretrained file:')
    for k, v in pretrained_dict.items():
        if k in model_dict:
            tmp[k] = v
        else:
            print(k)
    print('---------------------------')
    print('Weights not loaded into new model:')
    for k, v in model_dict.items():
        if k not in pretrained_dict:
            print(k)
    print('===================================\n')
    del pretrained_dict
    model_dict.update(tmp)
    del tmp
    model.load_state_dict(model_dict)
    return model

print('ResNet 2D3D backbone defined.')

## 7. Mô hình FGI (Audio-Visual)

In [ ]:
class My_CNN_RawAud(nn.Module):
    def __init__(self):
        super(My_CNN_RawAud, self).__init__()
        self.netcnnaud_layer1 = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=80, stride=8),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
        )
        self.netcnnaud_layer2 = nn.Sequential(
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Conv1d(128, 192, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(192),
            nn.ReLU(inplace=True),
            nn.Conv1d(192, 192, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(192),
        )

    def forward(self, x):
        x1 = self.netcnnaud_layer1(x)
        x2 = self.netcnnaud_layer2(x1)
        return x2


class My_Network(nn.Module):
    def __init__(self, network='resnet18', with_attention=False,
                 residual_conn=False, spatial_size=28):
        super(My_Network, self).__init__()
        self.__nFeatures__ = 24
        self.__nChs__ = 32
        self.__midChs__ = 32
        self.aud2visspace = None
        self.with_attention = with_attention
        self.residual_conn = residual_conn
        self.spatial_size = spatial_size

        self.netcnnaud = My_CNN_RawAud()

        a_size = [192, 1497]
        v_size = [128, 15, spatial_size, spatial_size]
        stride = int(a_size[1] / v_size[1])
        kernel_size = int(a_size[1] - (v_size[1] - 1) * stride)
        self.netcnnaud_to_vis = nn.Sequential(
            nn.Conv1d(a_size[0], v_size[0] // 2, kernel_size=kernel_size, stride=stride),
            nn.BatchNorm1d(v_size[0] // 2),
            nn.ReLU(),
            nn.Conv1d(v_size[0] // 2, v_size[0], kernel_size=3, stride=1, padding=1)
        )

        self.netcnnlip, self.param = select_resnet_half(network, track_running_stats=True)

        map_size = v_size[2] * v_size[3]
        self.final_fc = nn.Sequential(
            nn.Linear(map_size, 1),
            nn.Sigmoid()
        )

        out_emb = v_size[0] // 4
        self.img_emb_layer = nn.Conv3d(v_size[0], out_emb, 1)
        self.aud_emb_layer = nn.Conv1d(v_size[0], out_emb, 1)

    def forward_aud(self, x):
        (B, C) = x.shape
        x = x.view(B, 1, C)
        x = self.netcnnaud(x)
        x = self.netcnnaud_to_vis(x)
        return x

    def forward_lip(self, x):
        (B, N, C, NF, H, W) = x.shape
        x = x.view(B * N, C, NF, H, W)
        x = self.netcnnlip(x, return_intermediate=False)
        if self.spatial_size != 28:
            x = nn.functional.adaptive_avg_pool3d(x, (15, self.spatial_size, self.spatial_size))
        return x

    def forward(self, vid_seq, aud_seq):
        vid_out = self.forward_lip(vid_seq)
        aud_out = self.forward_aud(aud_seq)

        aud_out = aud_out.view(aud_out.shape[0], aud_out.shape[1],
                               aud_out.shape[2], 1, 1)

        vid_aud_distance_ = torch.pow((vid_out - aud_out), 2)
        vid_aud_distance_ = vid_aud_distance_.view(
            vid_aud_distance_.shape[0],
            vid_aud_distance_.shape[1] * vid_aud_distance_.shape[2],
            vid_aud_distance_.shape[3] * vid_aud_distance_.shape[4]
        )
        vid_aud_distance_ = torch.sqrt(torch.sum(vid_aud_distance_, dim=1))

        if self.with_attention:
            img_emb = self.img_emb_layer(vid_out)
            aud_emb = self.aud_emb_layer(
                aud_out.view(aud_out.shape[0], aud_out.shape[1], aud_out.shape[2])
            )
            atts = []
            for i_emb, a_emb in zip(img_emb, aud_emb):
                atts.append(torch.tensordot(i_emb, a_emb, dims=([0, 1], [0, 1])))
            att = torch.stack(atts)
            att = att / (32 * 15)
            att = att.view(att.shape[0], -1)
            att = torch.nn.functional.softmax(att, dim=1)

            if self.residual_conn:
                vid_aud_distance_ = torch.mul(att, vid_aud_distance_) + vid_aud_distance_
            else:
                vid_aud_distance_ = torch.mul(att, vid_aud_distance_)
        else:
            att = None

        final_out = self.final_fc(vid_aud_distance_)
        return final_out, vid_aud_distance_, att


print('FGI Model defined.')
_model = My_Network(with_attention=True)
total_params = sum(p.numel() for p in _model.parameters())
print(f'Total parameters: {total_params:,}')
del _model

## 8. Dataset

In [ ]:
def pil_loader(path):
    with open(path, 'rb') as f:
        with Image.open(f) as img:
            return img.convert('RGB')


def my_collate_rawaudio(batch):
    batch = list(filter(lambda x: x is not None and x[1].size()[0] == 48000, batch))
    if len(batch) == 0:
        return [[], [], [], [], [], [], []]
    return torch.utils.data.dataloader.default_collate(batch)


class deepfake_3d_rawaudio(data.Dataset):
    def __init__(self, out_dir, mode='train', transform=None,
                 vis_min_fake_len=2, vis_max_fake_len=-1,
                 aud_min_fake_len=2, aud_max_fake_len=-1,
                 using_pseudo_fake=False, dataset_name='dfdc',
                 pseudo_fake_prob=0.5, augment=False,
                 audio_aug_prob=0.0, audio_aug_fixed=False,
                 music_snr_db=(0, 20), noise_snr_db=(5, 25)):
        assert dataset_name in ['dfdc', 'fakeavceleb']
        self.mode = mode
        self.transform = transform
        self.out_dir = out_dir
        self.dataset_name = dataset_name

        if mode == 'train':
            split = os.path.join(self.out_dir, 'train_split.csv')
            video_info = pd.read_csv(split, header=None)
        elif mode == 'valid':
            split = os.path.join(self.out_dir, 'valid_split.csv')
            video_info = pd.read_csv(split, header=None)
        elif mode == 'test':
            split = os.path.join(self.out_dir, 'test_imbalance_split.csv')
            video_info = pd.read_csv(split, header=None)
        else:
            raise ValueError('wrong mode')

        self.using_pseudo_fake = using_pseudo_fake if mode == 'train' else False
        self.pseudo_fake_prob = pseudo_fake_prob
        self.augment = augment if mode == 'train' else False
        # audio aug: train -> ngau nhien theo audio_aug_prob; audio_aug_fixed=True (valid) -> luon ap dung, co dinh theo index
        self.audio_aug_fixed = audio_aug_fixed
        self.audio_aug_prob = audio_aug_prob if (mode == 'train' or audio_aug_fixed) else 0.0
        self.music_snr_db = music_snr_db
        self.noise_snr_db = noise_snr_db

        self.label_dict_encode = {'fake': 0, 'real': 1}
        self.label_dict_decode = {'0': 'fake', '1': 'real'}

        self.video_info = video_info
        self.vis_min_fake_len = vis_min_fake_len
        self.vis_max_fake_len = vis_max_fake_len
        self.aud_min_fake_len = aud_min_fake_len
        self.aud_max_fake_len = aud_max_fake_len

    def _generate_pseudo_fake(self, t_seq, audio, other_t_seq, other_audio):
        chosen_type = random.choice([0, 1, 2])
        if chosen_type == 0:
            audio = self._augment_pseudo_fake(audio, audio.shape[0],
                                              minimum_fake_length=self.aud_min_fake_len,
                                              maximum_fake_length=self.aud_max_fake_len,
                                              other_data=other_audio)
        elif chosen_type == 1:
            t_seq = self._augment_pseudo_fake(t_seq, t_seq.shape[0],
                                              minimum_fake_length=self.vis_min_fake_len,
                                              maximum_fake_length=self.vis_max_fake_len,
                                              other_data=other_t_seq)
        elif chosen_type == 2:
            audio = self._augment_pseudo_fake(audio, audio.shape[0],
                                              minimum_fake_length=self.aud_min_fake_len,
                                              maximum_fake_length=self.aud_max_fake_len,
                                              other_data=other_audio)
            t_seq = self._augment_pseudo_fake(t_seq, t_seq.shape[0],
                                              minimum_fake_length=self.vis_min_fake_len,
                                              maximum_fake_length=self.vis_max_fake_len,
                                              other_data=other_t_seq)
        return t_seq, audio, chosen_type

    def _select_pseudo_fake_window(self, data_length, minimum=2, maximum=-1):
        if 0 < minimum <= 1:
            minimum = max(2, int(minimum * data_length))
        if minimum == -1:
            minimum = data_length
        if maximum == -1:
            maximum = data_length
        elif 0 < maximum <= 1:
            maximum = min(minimum, int(maximum * data_length))
        assert minimum >= 2
        assert maximum >= minimum
        fake_len = random.randint(minimum, maximum)
        start_pos = random.randint(0, data_length - fake_len)
        end_pos = min(data_length, start_pos + fake_len)
        return fake_len, start_pos, end_pos

    def _replace_with_other(self, data_tensor, fake_len, start_pos, end_pos, other_data):
        data_tensor[start_pos:end_pos] = other_data[start_pos:end_pos]
        return data_tensor

    def _augment_pseudo_fake(self, data_tensor, time_len,
                             minimum_fake_length=2, maximum_fake_length=-1,
                             other_data=None):
        fake_len, start_pos, end_pos = self._select_pseudo_fake_window(
            data_length=time_len, minimum=minimum_fake_length, maximum=maximum_fake_length)
        data_tensor = self._replace_with_other(data_tensor, fake_len, start_pos, end_pos, other_data)
        return data_tensor

    def _get_other_item(self, index):
        if self.using_pseudo_fake and self.mode == 'train':
            success = 0
            while success == 0:
                try:
                    other_index = random.randint(0, self.__len__() - 1)
                    while index == other_index:
                        other_index = random.randint(0, self.__len__() - 1)
                    other_vpath, other_audiopath, other_label = self.video_info.iloc[other_index]
                    other_vpath = os.path.join(self.out_dir, other_vpath)
                    other_audiopath = os.path.join(self.out_dir, other_audiopath)
                    other_seq = [pil_loader(os.path.join(other_vpath, img)) for img in
                                 sorted(os.listdir(other_vpath))]
                    other_sample_rate, other_audio = wavfile.read(other_audiopath)
                    other_t_seq = self.transform(other_seq)
                    other_t_seq = torch.stack(other_t_seq, 0)
                    other_normalized_raw_audio = min_max_normalize(
                        other_audio, int(other_audio.min()), int(other_audio.max()))
                    other_normalized_raw_audio = torch.from_numpy(
                        other_normalized_raw_audio.astype(float)).float()
                    other_normalized_raw_audio = (other_normalized_raw_audio - 0.5) / 0.5
                    if len(other_normalized_raw_audio) < 48000 or len(other_t_seq) < 30:
                        continue
                    success = 1
                except:
                    continue
        else:
            other_t_seq = None
            other_normalized_raw_audio = None
        return other_t_seq, other_normalized_raw_audio

    def _augment_clip(self, t_seq):
        """Augmentation CHUNG cho ca 30 frame, ap dung SAU khi tron pseudo-fake
        (de doan frame lay tu video khac khong bi lech mau / lat so voi phan con lai,
        tranh viec model hoc "dau vet augmentation" thay vi lech hinh-tieng).
        t_seq: [T, C, H, W], da Normalize(0.5, 0.5) nen nam trong [-1, 1]."""
        if random.random() < 0.5:                       # lat ngang (khong anh huong dong bo hinh-tieng)
            t_seq = torch.flip(t_seq, dims=[-1])
        if random.random() < 0.8:                       # doi sang / tuong phan / bao hoa nhe
            x = t_seq * 0.5 + 0.5                       # ve [0, 1]
            fb = 1 + random.uniform(-0.2, 0.2)
            fc = 1 + random.uniform(-0.2, 0.2)
            fs = 1 + random.uniform(-0.2, 0.2)
            x = x * fb
            m = x.mean()
            x = (x - m) * fc + m
            gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
            x = (x - gray) * fs + gray
            t_seq = x.clamp(0, 1) * 2 - 1
        return t_seq

    def _synth_music(self, rng, n=48000, sr=48000):
        """Nhac tong hop don gian: hop am (gam truong/thu) + hoa am + envelope tat dan + (tuy chon) trong."""
        t = np.arange(n, dtype=np.float32) / sr
        out = np.zeros(n, dtype=np.float32)
        root = rng.choice([110.0, 130.8, 146.8, 164.8, 196.0, 220.0, 261.6])
        scale = [0, 2, 4, 5, 7, 9, 11, 12] if rng.random() < 0.5 else [0, 2, 3, 5, 7, 8, 10, 12]
        seg = rng.uniform(0.15, 0.5)
        start = 0.0
        while start < n / sr:
            for d in rng.sample(scale, rng.randint(1, 3)):
                f = root * 2 ** ((d + 12 * rng.choice([0, 1])) / 12.0)
                tt = np.clip(t - start, 0, None)
                env = np.where(t >= start, np.exp(-tt * rng.uniform(2.0, 6.0)), 0.0)
                wave = sum(np.sin(2 * np.pi * f * k * tt) / k for k in (1, 2, 3, 4))
                out += (env * wave).astype(np.float32)
            start += seg
        if rng.random() < 0.5:                      # tieng trong / beat
            period = rng.choice([0.25, 0.5])
            for s0 in np.arange(0, n / sr, period):
                tt = np.clip(t - s0, 0, None)
                out += (np.where(t >= s0, 1.5 * np.exp(-tt * 20) * np.sin(2 * np.pi * 60 * tt), 0.0)).astype(np.float32)
        return out

    def _augment_audio(self, a, rng):
        """a: tensor [48000] trong [-1, 1] (da min-max). Chen nhac/noise theo SNR roi min-max lai
        de giu dung phan bo dau vao nhu luc train/suy luan."""
        x = a.numpy().astype(np.float32)
        x = x - x.mean()
        rms = float(np.sqrt(np.mean(x ** 2))) + 1e-8
        kind = rng.choice(['music', 'music', 'noise', 'both'])
        mix = np.zeros_like(x)
        if kind in ('music', 'both'):
            m = self._synth_music(rng, len(x))
            m = m / (float(np.sqrt(np.mean(m ** 2))) + 1e-8) * rms
            mix += m * 10 ** (-rng.uniform(*self.music_snr_db) / 20.0)
        if kind in ('noise', 'both'):
            rs = np.random.RandomState(rng.randint(0, 2 ** 31 - 1))
            w = rs.randn(len(x)).astype(np.float32)
            if rng.random() < 0.5:                  # pink noise
                sp = np.fft.rfft(w)
                f = np.arange(len(sp), dtype=np.float32)
                f[0] = 1.0
                w = np.fft.irfft(sp / np.sqrt(f), len(x)).astype(np.float32)
            w = w / (float(np.sqrt(np.mean(w ** 2))) + 1e-8) * rms
            mix += w * 10 ** (-rng.uniform(*self.noise_snr_db) / 20.0)
        y = x + mix
        y = (y - y.min()) / (y.max() - y.min() + 1e-8) * 2.0 - 1.0
        return torch.from_numpy(y).float()

    def __getitem__(self, index):
        success = 0
        while success == 0:
            try:
                vpath, audiopath, label = self.video_info.iloc[index]
                vpath = os.path.join(self.out_dir, vpath)
                audiopath = os.path.join(self.out_dir, audiopath)
                seq = [pil_loader(os.path.join(vpath, img)) for img in sorted(os.listdir(vpath))]
                sample_rate, audio = wavfile.read(audiopath)
                t_seq = self.transform(seq)
                (C, H, W) = t_seq[0].size()
                t_seq = torch.stack(t_seq, 0)
                normalized_raw_audio = min_max_normalize(audio, int(audio.min()), int(audio.max()))
                normalized_raw_audio = torch.from_numpy(normalized_raw_audio.astype(float)).float()
                normalized_raw_audio = (normalized_raw_audio - 0.5) / 0.5
                success = 1
            except:
                index = random.randint(0, self.__len__() - 1)
                continue

        if self.using_pseudo_fake:
            use_pseudo_fake = random.random() < self.pseudo_fake_prob
            if use_pseudo_fake:
                other_t_seq, other_normalized_raw_audio = self._get_other_item(index)
                t_seq, normalized_raw_audio, chosen_type = self._generate_pseudo_fake(
                    t_seq, normalized_raw_audio, other_t_seq, other_normalized_raw_audio)
                label = 'fake'

        if self.audio_aug_prob > 0:
            _rng = random.Random(12345 + int(index)) if self.audio_aug_fixed else random
            if _rng.random() < self.audio_aug_prob:
                normalized_raw_audio = self._augment_audio(normalized_raw_audio, _rng)

        if self.augment:
            t_seq = self._augment_clip(t_seq)

        t_seq = t_seq.view(1, 30, C, H, W).transpose(1, 2)
        vid = self.label_dict_encode[label]
        return t_seq, normalized_raw_audio, torch.LongTensor([vid]), audiopath

    def __len__(self):
        return len(self.video_info)

    def encode_label(self, label_name):
        return self.label_dict_encode[label_name]

print('Dataset class defined.')

## 9. Data Loaders

In [ ]:
def get_rawaudio_data(transform, cfg, mode='train', music_fixed=False):
    print(f'Loading data for "{mode}"...')
    dataset = deepfake_3d_rawaudio(
        out_dir=cfg.WORK_DIR, mode=mode, transform=transform,
        vis_min_fake_len=cfg.VIS_MIN_FAKE_LEN, vis_max_fake_len=cfg.VIS_MAX_FAKE_LEN,
        aud_min_fake_len=cfg.AUD_MIN_FAKE_LEN, aud_max_fake_len=cfg.AUD_MAX_FAKE_LEN,
        using_pseudo_fake=cfg.USING_PSEUDO_FAKE if mode == 'train' else False,
        dataset_name='dfdc',
        pseudo_fake_prob=getattr(cfg, 'PSEUDO_FAKE_PROB', 0.5),
        augment=(mode == 'train' and getattr(cfg, 'USE_AUGMENTATION', False)),
        audio_aug_prob=(1.0 if music_fixed else (getattr(cfg, 'AUDIO_AUG_PROB', 0.0) if mode == 'train' else 0.0)),
        audio_aug_fixed=music_fixed,
        music_snr_db=getattr(cfg, 'MUSIC_SNR_DB', (0, 20)),
        noise_snr_db=getattr(cfg, 'NOISE_SNR_DB', (5, 25))
    )
    # Giu worker song giua cac epoch + nap truoc nhieu batch (chi cho train/valid; test chi chay 1 lan)
    loader_extra = {}
    if cfg.NUM_WORKERS > 0 and mode in ('train', 'valid') and not music_fixed:
        loader_extra = dict(persistent_workers=True, prefetch_factor=cfg.PREFETCH_FACTOR)
    sampler = data.RandomSampler(dataset)

    if mode == 'train':
        data_loader = data.DataLoader(
            dataset, batch_size=cfg.BATCH_SIZE, sampler=sampler, shuffle=False,
            num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
            collate_fn=my_collate_rawaudio, **loader_extra
        )
    elif mode in ('valid', 'test'):
        data_loader = data.DataLoader(
            dataset, batch_size=1, sampler=sampler, shuffle=False,
            num_workers=cfg.NUM_WORKERS, pin_memory=True,
            collate_fn=my_collate_rawaudio, **loader_extra
        )
    else:
        raise ValueError(f'No mode {mode}')

    print(f'"{mode}" dataset size: {len(dataset)}')
    return data_loader


train_transform = transforms.Compose([
    Scale(size=(cfg.IMG_DIM, cfg.IMG_DIM)),
    ToTensor(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    Scale(size=(cfg.IMG_DIM, cfg.IMG_DIM)),
    ToTensor(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_loader = get_rawaudio_data(train_transform, cfg, 'train')
print(f'Train loader: {len(train_loader)} batches')

# Validation loader: used for per-epoch checkpointing (pseudo-fake augmentation disabled)
valid_cfg = copy.deepcopy(cfg)
valid_cfg.USING_PSEUDO_FAKE = False
valid_loader = get_rawaudio_data(test_transform, valid_cfg, 'valid')
print(f'Valid loader: {len(valid_loader)} samples')

# Valid "co chen nhac/noise" (co dinh theo index): do muc do ben vung voi nhac nen
valid_music_loader = None
if cfg.EVAL_MUSIC_VALID:
    valid_music_loader = get_rawaudio_data(test_transform, valid_cfg, 'valid', music_fixed=True)
    print(f'Valid-music loader: {len(valid_music_loader)} samples')


## 10. Hàm train / test

In [ ]:
def train_one_epoch(data_loader, model, optimizer, criterion, epoch, cfg, writer=None):
    losses = AverageMeter()
    real_distances = AverageMeter()
    fake_distances = AverageMeter()
    model.train()
    cuda_dev = torch.device('cuda')

    for idx, batch_data in enumerate(data_loader):
        if len(batch_data[0]) == 0:
            continue
        video_seq, audio_seq, target, audiopath = batch_data
        target = 1 - target

        tic = time.time()
        video_seq = video_seq.to(cuda_dev)
        audio_seq = audio_seq.to(cuda_dev)
        target = target.to(cuda_dev)
        B = video_seq.size(0)

        out, vid_out_dist, att = model(video_seq, audio_seq)
        del video_seq, audio_seq

        loss = criterion(out, target.float())
        losses.update(loss.item(), B)

        n_real = torch.sum(target.view(-1) == 0).detach().cpu().item()
        if n_real > 0:
            real_distances.update(torch.mean(vid_out_dist[target.view(-1) == 0]).item())
        n_fake = torch.sum(target.view(-1) == 1).detach().cpu().item()
        if n_fake > 0:
            fake_distances.update(torch.mean(vid_out_dist[target.view(-1) == 1]).item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if idx % cfg.PRINT_FREQ == 0:
            elapsed = time.time() - tic
            print(f'Epoch [{epoch}][{idx}/{len(data_loader)}] '
                  f'Time: {elapsed:.2f}s  '
                  f'Loss: {losses.val:.4f} ({losses.local_avg:.4f})  '
                  f'Real: {real_distances.val:.4f}  Fake: {fake_distances.val:.4f}')

        if writer:
            writer.add_scalar('local/loss', losses.val, epoch * len(data_loader) + idx)

    return losses.avg, real_distances.avg, fake_distances.avg


def test_model(data_loader, model, cuda_device, threshold=0.5, return_details=False):
    real_distances = AverageMeter()
    fake_distances = AverageMeter()
    model.eval()
    test_pred = {}
    test_target = {}
    test_num_chunks = {}

    with torch.no_grad():
        for idx, batch_data in tqdm(enumerate(data_loader), total=len(data_loader), desc='Testing'):
            if len(batch_data[0]) == 0:
                continue
            video_seq, audio_seq, target, audiopath = batch_data
            target = 1 - target

            video_seq = video_seq[0].unsqueeze(0).to(cuda_device)
            audio_seq = audio_seq[0].unsqueeze(0).to(cuda_device)
            target = target[0].unsqueeze(0).to(cuda_device)

            pred, vid_aud_dist, att = model(video_seq, audio_seq)
            del video_seq, audio_seq

            tar = target[0, :].view(-1).item()
            vid_name = audiopath[0].split('/')[-2]

            if test_pred.get(vid_name):
                test_pred[vid_name] += pred[0].view(-1).item()
                test_num_chunks[vid_name] += 1
            else:
                test_pred[vid_name] = pred[0].view(-1).item()
                test_num_chunks[vid_name] = 1

            if not test_target.get(vid_name):
                test_target[vid_name] = tar

            if tar == 1:
                fake_distances.update(torch.mean(vid_aud_dist[0]).item())
            else:
                real_distances.update(torch.mean(vid_aud_dist[0]).item())

    pred_fake = []
    pred_real = []
    for video, score in test_pred.items():
        tar = test_target[video]
        num_chunks = test_num_chunks[video]
        mean_pred = score / num_chunks
        if tar == 1:
            pred_fake.append(mean_pred)
        else:
            pred_real.append(mean_pred)

    if len(pred_fake) > 0 and len(pred_real) > 0:
        pred_all = np.concatenate([np.array(pred_fake), np.array(pred_real)])
        gt_all = np.concatenate([np.ones(len(pred_fake)), np.zeros(len(pred_real))])
        auc = roc_auc_score(gt_all, pred_all)
        pred_label = (pred_all >= threshold).astype(int)
        accuracy = accuracy_score(gt_all, pred_label)
        precision = precision_score(gt_all, pred_label)
        recall = recall_score(gt_all, pred_label)
        f1 = f1_score(gt_all, pred_label)
        cm = confusion_matrix(gt_all, pred_label, labels=[0, 1])
    else:
        pred_all = np.array([])
        gt_all = np.array([])
        pred_label = np.array([])
        auc = accuracy = precision = recall = f1 = 0.0
        cm = np.zeros((2, 2), dtype=int)

    print(f'Fake dist: {fake_distances.avg:.4f}  Real dist: {real_distances.avg:.4f}  '
          f'AUC: {auc:.4f}  Acc: {accuracy:.4f}  Precision: {precision:.4f}  '
          f'Recall: {recall:.4f}  F1: {f1:.4f}')

    if return_details:
        details = {
            'y_true': gt_all,
            'y_score': pred_all,
            'y_pred': pred_label,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'confusion_matrix': cm,
        }
        return auc, real_distances.avg, fake_distances.avg, details

    return auc, real_distances.avg, fake_distances.avg

print('Training and testing functions defined.')

## 11. Khởi tạo model (tự resume nếu có checkpoint)

Nếu có `epoch*.pth.tar` trong `MyDrive/pbl6/models_<RUN_TAG>` thì nạp lại trọng số, optimizer, số epoch và trạng thái chọn best (`best_score`...). Đổi `cfg.RUN_TAG` để train mới với công thức mới; muốn train lại từ đầu với cùng tag: xóa các file trong thư mục `models_<RUN_TAG>`.

In [ ]:
# ====================== INITIALIZE MODEL ======================

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
cuda = torch.device('cuda')

model = My_Network(
    with_attention=cfg.WITH_ATTENTION,
    residual_conn=cfg.RESIDUAL_CONN,
    spatial_size=cfg.SPATIAL_SIZE
)
model = model.cuda()
model = nn.DataParallel(model)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
# Giam LR khi SELECT SCORE khong tang (step sau moi lan validate, xem vong train)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=cfg.LR_FACTOR, patience=cfg.LR_PATIENCE, min_lr=cfg.MIN_LR)

log_dir = os.path.join(cfg.DRIVE_DIR, f'logs_{cfg.RUN_TAG}')
model_dir = os.path.join(cfg.DRIVE_DIR, f'models_{cfg.RUN_TAG}')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)
writer = SummaryWriter(logdir=log_dir)

print(f'RUN_TAG={cfg.RUN_TAG} -> models: {model_dir}')
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable_params:,} / Total: {total_params:,} parameters')

# ---- Resume tu checkpoint moi nhat tren Drive (neu co) ----
start_epoch = 0
resume_state = None
ckpts = glob.glob(os.path.join(model_dir, 'epoch*.pth.tar'))
if ckpts:
    latest = max(ckpts, key=lambda p: int(re.search(r'epoch(\d+)\.pth\.tar', p).group(1)))
    resume_state = torch.load(latest, map_location='cuda', weights_only=False)
    model.load_state_dict(resume_state['state_dict'])
    optimizer.load_state_dict(resume_state['optimizer'])
    if 'scheduler' in resume_state:
        scheduler.load_state_dict(resume_state['scheduler'])
    start_epoch = resume_state['epoch']
    print(f'Resume tu {latest} -> tiep tuc tu epoch {start_epoch + 1}')
else:
    print('Khong co checkpoint, train tu dau.')

## 12. Vòng lặp huấn luyện chính

In [ ]:
!pip install -q tensorboard
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(log_dir, size_guidance={'scalars': 0})
ea.Reload()

def get(tag):
    return [(e.step, e.value) for e in ea.Scalars(tag)] if tag in ea.Tags()['scalars'] else []

train_losses     = [v for _, v in get('global/loss')]
train_real_dists = [v for _, v in get('global/real_distance')]
train_fake_dists = [v for _, v in get('global/fake_distance')]
valid_aucs       = get('global/valid_auc')
valid_accuracies = get('global/valid_accuracy')
valid_precisions = get('global/valid_precision')
valid_recalls    = get('global/valid_recall')
valid_f1s        = get('global/valid_f1')

print(len(train_losses), 'epoch train;', len(valid_aucs), 'epoch valid')

In [ ]:
# ====================== MAIN TRAINING LOOP ======================
from torch.utils.data import Subset

# ---- Ham phu tro cho cong thuc chon best ----
def f1_opt_sweep(y_true, y_score):
    """F1 o nguong toi uu, quet 0.05..0.95 tren valid. Tra ve (f1_opt, nguong)."""
    thrs = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y_true, (y_score >= t).astype(int), zero_division=0) for t in thrs]
    k = int(np.argmax(f1s))
    return float(f1s[k]), float(thrs[k])


def selection_score(comp, gap):
    """score = base - phat. Tra ve (score, base, phat, day_du).
    day_du=False neu thieu thanh phan nao (None / NaN) hoac thieu gap.
    Key la trong SELECT_WEIGHTS -> bao loi, khong am tham bo qua."""
    unknown = [k for k in cfg.SELECT_WEIGHTS if k not in comp]
    if unknown:
        raise KeyError(f'SELECT_WEIGHTS co key khong ho tro: {unknown}; ho tro: {list(comp)}')
    wanted = {k: w for k, w in cfg.SELECT_WEIGHTS.items() if w > 0}
    bad = [k for k in wanted if comp[k] is None or not np.isfinite(comp[k])]
    if bad or gap is None or not np.isfinite(gap):
        return None, None, None, False
    base = sum(w * comp[k] for k, w in wanted.items()) / sum(wanted.values())
    penalty = cfg.GAP_PENALTY * max(0.0, gap - cfg.GAP_TOL)
    return base - penalty, base, penalty, True


if 'music_auc' in cfg.SELECT_WEIGHTS and not cfg.EVAL_MUSIC_VALID:
    raise ValueError("SELECT_WEIGHTS co 'music_auc' nhung EVAL_MUSIC_VALID=False")

# ---- Tap train "sach" co dinh (CLEAN_TRAIN_PER_CLASS video / lop; tat pseudo-fake + augmentation) ----
ds_clean = deepfake_3d_rawaudio(out_dir=cfg.WORK_DIR, mode='train', transform=test_transform,
                                using_pseudo_fake=False, dataset_name='dfdc', augment=False)
_labels = ds_clean.video_info.iloc[:, 2].values
_rng = random.Random(cfg.SEED)
clean_idx = []
for _lab in ('real', 'fake'):
    _ids = [i for i, l in enumerate(_labels) if l == _lab]
    clean_idx += _rng.sample(_ids, min(cfg.CLEAN_TRAIN_PER_CLASS, len(_ids)))
clean_loader = data.DataLoader(Subset(ds_clean, clean_idx), batch_size=1, shuffle=False,
                               num_workers=cfg.NUM_WORKERS, pin_memory=True,
                               collate_fn=my_collate_rawaudio)
print(f'Train-sach: {len(clean_idx)} video ({cfg.CLEAN_TRAIN_PER_CLASS}/lop, co dinh theo SEED)')

print('=' * 60)
print('Starting Training')
print(f'  Epochs: {cfg.EPOCHS}, BS: {cfg.BATCH_SIZE}, LR: {cfg.LEARNING_RATE}')
print(f'  Attention: {cfg.WITH_ATTENTION}, Pseudo-Fake: {cfg.USING_PSEUDO_FAKE}')
print(f'  Chon best: score = base({cfg.SELECT_WEIGHTS}) - {cfg.GAP_PENALTY}*max(0, gap - {cfg.GAP_TOL}); '
      f'hop le khi gap <= {cfg.MAX_GAP_AUC}; delta >= {cfg.SCORE_MIN_DELTA}')
print('=' * 60)

least_loss = float('inf')
max_valid_auc = -1.0          # AUC cao nhat tung thay tren valid (de tuong thich)
best_model_auc = -1.0         # AUC cua model dang duoc chon la best
best_valid_logloss = float('inf')
best_valid_auc = -1.0         # giu ten cu de tuong thich voi checkpoint / cell 46
best_score = -float('inf')    # score cao nhat trong cac epoch HOP LE
best_fb_score = -float('inf') # du phong: score cao nhat khi chua co epoch hop le nao
has_eligible = False
best_epoch = None
train_losses = []
train_real_dists = []
train_fake_dists = []
valid_aucs = []
valid_accuracies = []
valid_precisions = []
valid_recalls = []
valid_f1s = []
valid_loglosses = []
sel_hist = []                 # lich su score tung epoch (de in bang tong hop)
no_improve = 0
overfit_streak = 0

if resume_state is not None:
    least_loss = resume_state['least_loss']
    max_valid_auc = resume_state.get('max_valid_auc', resume_state.get('best_valid_auc', -1.0))
    best_model_auc = resume_state.get('best_model_auc', -1.0)
    best_valid_logloss = resume_state.get('best_valid_logloss', float('inf'))
    best_valid_auc = max_valid_auc
    best_score = resume_state.get('best_score', -float('inf'))
    best_fb_score = resume_state.get('best_fb_score', -float('inf'))
    has_eligible = resume_state.get('has_eligible', False)
    best_epoch = resume_state.get('best_epoch', None)
    no_improve = resume_state.get('no_improve', 0)
    overfit_streak = resume_state.get('overfit_streak', 0)

torch.backends.cudnn.benchmark = True

for epoch in range(start_epoch, cfg.EPOCHS):
    print(f'\n--- Epoch {epoch+1}/{cfg.EPOCHS} ---')

    train_loss, real_distance, fake_distance = train_one_epoch(
        train_loader, model, optimizer, criterion, epoch, cfg, writer
    )

    train_losses.append(train_loss)
    train_real_dists.append(real_distance)
    train_fake_dists.append(fake_distance)

    writer.add_scalar('global/loss', train_loss, epoch)
    writer.add_scalar('global/real_distance', real_distance, epoch)
    writer.add_scalar('global/fake_distance', fake_distance, epoch)

    print(f'Epoch {epoch+1}: Loss={train_loss:.4f}, '
          f'Real={real_distance:.4f}, Fake={fake_distance:.4f}')

    least_loss = min(least_loss, train_loss)

    # ---- Validation: chon best model tren tap valid (muc video) ----
    run_valid = ((epoch + 1) % cfg.VALID_FREQ == 0) or (epoch == cfg.EPOCHS - 1)
    is_best = False
    valid_auc = None
    stop_reason = None
    if run_valid:
        valid_auc, valid_real_dist, valid_fake_dist, valid_metrics = test_model(
            valid_loader, model, cuda, return_details=True
        )
        # log loss + F1 o nguong toi uu (muc video)
        if len(valid_metrics['y_true']) > 0:
            y_t = valid_metrics['y_true'].astype(int)
            y_s = valid_metrics['y_score']
            valid_logloss = float(log_loss(y_t, np.clip(y_s, 1e-7, 1 - 1e-7), labels=[0, 1]))
            valid_f1_opt, valid_thr_opt = f1_opt_sweep(y_t, y_s)
        else:
            valid_logloss, valid_f1_opt, valid_thr_opt = float('inf'), 0.0, 0.5

        # AUC tren tap train sach (cung cach danh gia nhu valid)
        clean_auc, _, _, _ = test_model(clean_loader, model, cuda, return_details=True)
        gap = clean_auc - valid_auc

        # AUC tren valid bi chen nhac/noise (do ben vung am thanh)
        music_auc = None
        if cfg.EVAL_MUSIC_VALID:
            music_auc, _, _, _ = test_model(valid_music_loader, model, cuda, return_details=True)

        valid_aucs.append((epoch, valid_auc))
        valid_accuracies.append((epoch, valid_metrics['accuracy']))
        valid_precisions.append((epoch, valid_metrics['precision']))
        valid_recalls.append((epoch, valid_metrics['recall']))
        valid_f1s.append((epoch, valid_metrics['f1']))
        valid_loglosses.append((epoch, valid_logloss))

        # ---- Cong thuc chon best ----
        comp = {
            'auc':  valid_auc,
            'music_auc': music_auc,                                # (chi dung neu them vao SELECT_WEIGHTS)
            'f1':   valid_f1_opt,                                  # F1 o nguong toi uu
            'acc':  valid_metrics['accuracy'],                     # (chi dung neu them vao SELECT_WEIGHTS)
            'loss': 1.0 - min(valid_logloss, 1.0),                 # (chi dung neu them vao SELECT_WEIGHTS)
        }
        score, base, penalty, complete = selection_score(comp, gap)
        overfit = gap > cfg.MAX_GAP_AUC
        eligible = complete and (not overfit)

        if eligible and score > best_score + cfg.SCORE_MIN_DELTA:
            is_best, has_eligible, best_score = True, True, score
        elif complete and (not has_eligible) and score > best_fb_score:
            # du phong: chua co epoch hop le nao -> tam giu epoch co score cao nhat
            is_best, best_fb_score = True, score

        if is_best:
            best_model_auc = valid_auc
            best_valid_logloss = valid_logloss
            best_epoch = epoch + 1
            no_improve = 0
        else:
            no_improve += 1
        overfit_streak = overfit_streak + 1 if overfit else 0
        max_valid_auc = max(max_valid_auc, valid_auc)
        best_valid_auc = max_valid_auc

        sel_hist.append({
            'epoch': epoch + 1, 'AUC': valid_auc, 'F1_opt': valid_f1_opt, 'thr_opt': valid_thr_opt,
            'MusicAUC': music_auc, 'TrainAUC': clean_auc, 'Gap': gap, 'Base': base, 'Phat': penalty, 'Score': score,
            'Eligible': eligible, 'Best': is_best, 'LogLoss': valid_logloss,
            'F1@0.5': valid_metrics['f1'], 'Precision': valid_metrics['precision'],
            'Recall': valid_metrics['recall'], 'Acc@0.5': valid_metrics['accuracy'],
        })

        writer.add_scalar('global/valid_auc', valid_auc, epoch)
        writer.add_scalar('global/valid_accuracy', valid_metrics['accuracy'], epoch)
        writer.add_scalar('global/valid_precision', valid_metrics['precision'], epoch)
        writer.add_scalar('global/valid_recall', valid_metrics['recall'], epoch)
        writer.add_scalar('global/valid_f1', valid_metrics['f1'], epoch)
        writer.add_scalar('global/valid_f1_opt', valid_f1_opt, epoch)
        writer.add_scalar('global/valid_logloss', valid_logloss, epoch)
        writer.add_scalar('global/train_clean_auc', clean_auc, epoch)
        writer.add_scalar('global/gap_auc', gap, epoch)
        if music_auc is not None:
            writer.add_scalar('global/valid_music_auc', music_auc, epoch)
        if score is not None:
            writer.add_scalar('global/select_score', score, epoch)

        sc_txt = f'{score:.4f}' if score is not None else 'N/A'
        print(f'  Validation -> AUC: {valid_auc:.4f}  F1_opt: {valid_f1_opt:.4f} (thr {valid_thr_opt:.2f})  '
              f'LogLoss: {valid_logloss:.4f}  Acc@0.5: {valid_metrics["accuracy"]:.4f}  '
              f'F1@0.5: {valid_metrics["f1"]:.4f}')
        if music_auc is not None:
            print(f'  Valid + nhac/noise -> AUC: {music_auc:.4f}  (tut {valid_auc - music_auc:+.4f} so voi valid goc)')
        print(f'  Gap: train-sach AUC {clean_auc:.4f} - valid AUC {valid_auc:.4f} = {gap:+.4f}  |  '
              f'score = {sc_txt}  ({"hop le" if eligible else "KHONG hop le"}'
              f'{", OVERFIT" if overfit else ""})')

        # ---- LR scheduler: giam LR theo score ----
        if score is not None:
            scheduler.step(score)
        cur_lr = optimizer.param_groups[0]['lr']
        writer.add_scalar('global/lr', cur_lr, epoch)
        print(f'  LR hien tai: {cur_lr:.2e}')

    save_checkpoint({
        'epoch': epoch + 1,
        'net': cfg.NET,
        'state_dict': model.state_dict(),
        'least_loss': least_loss,
        'best_valid_auc': best_valid_auc,
        'max_valid_auc': max_valid_auc,
        'best_model_auc': best_model_auc,
        'best_valid_logloss': best_valid_logloss,
        'best_score': best_score,
        'best_fb_score': best_fb_score,
        'has_eligible': has_eligible,
        'best_epoch': best_epoch,
        'no_improve': no_improve,
        'overfit_streak': overfit_streak,
        'run_tag': cfg.RUN_TAG,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'iteration': epoch
    }, is_best, filename=os.path.join(model_dir, f'epoch{epoch+1}.pth.tar'),
       keep_all=cfg.SAVE_ALL)

    if is_best:
        tag = '' if has_eligible else ' [du phong - chua co epoch hop le]'
        print(f'  * New best model!{tag} (score={score:.4f}, valid AUC={valid_auc:.4f}, F1_opt={valid_f1_opt:.4f})')

    if cfg.EARLY_STOP_PATIENCE is not None and no_improve >= cfg.EARLY_STOP_PATIENCE:
        print(f'  Early stopping: {no_improve} epoch lien tiep khong co best model moi.')
        break
    if (cfg.OVERFIT_STOP_PATIENCE is not None and overfit_streak >= cfg.OVERFIT_STOP_PATIENCE
            and not is_best):
        print(f'  Dung vi overfit: gap > {cfg.MAX_GAP_AUC} lien tiep {overfit_streak} epoch va khong co best moi.')
        break

_bs = best_score if has_eligible else best_fb_score
print(f'\nTraining finished! Best model: epoch {best_epoch}, score={_bs:.4f}, valid AUC={best_model_auc:.4f}, '
      f'logloss={best_valid_logloss:.4f} (AUC cao nhat: {max_valid_auc:.4f})'
      + ('' if has_eligible else '  [CANH BAO: khong co epoch hop le, dang dung best du phong]'))
writer.close()

# ---- Bang tong hop cac epoch tren valid de doi chieu chon best model ----
if sel_hist:
    summary_val = pd.DataFrame(sel_hist)
    print('\nBang valid theo epoch (best = epoch hop le co score cao nhat, hon best truoc >= SCORE_MIN_DELTA):')
    print(summary_val.round(4).to_string(index=False))
    summary_val.to_csv(os.path.join(cfg.WORK_DIR, 'valid_metrics_per_epoch.csv'), index=False)


## 13. Biểu đồ huấn luyện

In [ ]:
!pip install -q tensorboard
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(log_dir, size_guidance={'scalars': 0})
ea.Reload()

def get(tag):
    return [(e.step, e.value) for e in ea.Scalars(tag)] if tag in ea.Tags()['scalars'] else []

train_losses     = [v for _, v in get('global/loss')]
train_real_dists = [v for _, v in get('global/real_distance')]
train_fake_dists = [v for _, v in get('global/fake_distance')]
valid_aucs       = get('global/valid_auc')
valid_accuracies = get('global/valid_accuracy')
valid_precisions = get('global/valid_precision')
valid_recalls    = get('global/valid_recall')
valid_f1s        = get('global/valid_f1')

print(len(train_losses), 'epoch train;', len(valid_aucs), 'epoch valid')

In [ ]:
# ====================== KIỂM TRA OVERFITTING (1 CELL) ======================
from torch.utils.data import Subset

# 1) Đo train "sạch": tắt pseudo-fake + augmentation, cùng cách đánh giá như valid
ds_tr = deepfake_3d_rawaudio(out_dir=cfg.WORK_DIR, mode='train', transform=test_transform,
                             using_pseudo_fake=False, dataset_name='dfdc', augment=False)
n = min(len(ds_tr), len(valid_loader.dataset))          # cùng cỡ với valid
random.seed(cfg.SEED)
idx = random.sample(range(len(ds_tr)), n)
tr_eval_loader = data.DataLoader(Subset(ds_tr, idx), batch_size=1, shuffle=False,
                                 num_workers=cfg.NUM_WORKERS, collate_fn=my_collate_rawaudio)

tr_auc, _, _, tr_m = test_model(tr_eval_loader, model, cuda, return_details=True)
tr_ll = log_loss(tr_m['y_true'].astype(int),
                 np.clip(tr_m['y_score'], 1e-7, 1 - 1e-7), labels=[0, 1])

# 2) Số liệu valid (epoch tốt nhất) và test
best_v_auc = max(a for _, a in valid_aucs)
best_v_ll  = min(l for _, l in valid_loglosses)

print('=' * 60)
print(f'TRAIN (sạch) : AUC={tr_auc:.4f}  logloss={tr_ll:.4f}  F1={tr_m["f1"]:.4f}')
print(f'VALID (best) : AUC={best_v_auc:.4f}  logloss={best_v_ll:.4f}')
print(f'TEST         : AUC={auc:.4f}  F1={test_metrics["f1"]:.4f}')
print('-' * 60)
gap_tv = tr_auc - best_v_auc
gap_vt = best_v_auc - auc
print(f'Gap AUC train - valid: {gap_tv:+.4f}')
print(f'Gap AUC valid - test : {gap_vt:+.4f}')

if gap_tv > 0.10:
    print('=> OVERFIT RÕ: train cao hơn valid nhiều.')
elif gap_tv > 0.05:
    print('=> Bắt đầu overfit (train cao hơn valid 0.05-0.10).')
elif gap_tv > 0.03:
    print('=> Hơi overfit nhẹ, chấp nhận được.')
else:
    print('=> Không thấy dấu hiệu overfit đáng kể giữa train và valid.')
if gap_vt > 0.03:
    print('=> Valid cao hơn test đáng kể: best epoch/threshold có thể bị "khớp" với valid.')

# 3) Biểu đồ: train loss vs valid logloss + valid AUC vs train AUC
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(range(1, len(train_losses) + 1), train_losses, 'b-o', label='Train loss (có pseudo-fake/aug)')
ax[0].plot([e + 1 for e, _ in valid_loglosses], [l for _, l in valid_loglosses],
           'r-o', label='Valid logloss')
ax[0].set_title('Loss theo epoch (valid tăng lại = overfit)')
ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot([e + 1 for e, _ in valid_aucs], [a for _, a in valid_aucs], 'c-o', label='Valid AUC')
ax[1].axhline(tr_auc, color='b', ls='--', label=f'Train AUC

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_real_dists, 'g-', linewidth=2, label='Real')
axes[1].plot(train_fake_dists, 'r-', linewidth=2, label='Fake')
axes[1].set_title('Audio-Visual Distance', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Distance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

gap = [f - r for f, r in zip(train_fake_dists, train_real_dists)]
axes[2].plot(gap, 'm-', linewidth=2)
axes[2].set_title('Distance Gap (Fake - Real)', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Gap')
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2].grid(True, alpha=0.3)

if valid_aucs:
    valid_epochs = [e for e, _ in valid_aucs]
    valid_auc_vals = [a for _, a in valid_aucs]
    axes[3].plot(valid_epochs, valid_auc_vals, 'c-o', linewidth=2, markersize=4)
axes[3].set_title('Validation AUC', fontsize=14)
axes[3].set_xlabel('Epoch')
axes[3].set_ylabel('AUC')
axes[3].set_ylim(0, 1)
axes[3].grid(True, alpha=0.3)

if valid_accuracies:
    acc_epochs = [e for e, _ in valid_accuracies]
    acc_vals = [a for _, a in valid_accuracies]
    axes[4].plot(acc_epochs, acc_vals, 'b-o', linewidth=2, markersize=4)
axes[4].set_title('Validation Accuracy', fontsize=14)
axes[4].set_xlabel('Epoch')
axes[4].set_ylabel('Accuracy')
axes[4].set_ylim(0, 1)
axes[4].grid(True, alpha=0.3)

if valid_precisions:
    prec_epochs = [e for e, _ in valid_precisions]
    prec_vals = [a for _, a in valid_precisions]
    axes[5].plot(prec_epochs, prec_vals, 'g-o', linewidth=2, markersize=4)
axes[5].set_title('Validation Precision', fontsize=14)
axes[5].set_xlabel('Epoch')
axes[5].set_ylabel('Precision')
axes[5].set_ylim(0, 1)
axes[5].grid(True, alpha=0.3)

if valid_recalls:
    rec_epochs = [e for e, _ in valid_recalls]
    rec_vals = [a for _, a in valid_recalls]
    axes[6].plot(rec_epochs, rec_vals, 'r-o', linewidth=2, markersize=4)
axes[6].set_title('Validation Recall', fontsize=14)
axes[6].set_xlabel('Epoch')
axes[6].set_ylabel('Recall')
axes[6].set_ylim(0, 1)
axes[6].grid(True, alpha=0.3)

if valid_f1s:
    f1_epochs = [e for e, _ in valid_f1s]
    f1_vals = [a for _, a in valid_f1s]
    axes[7].plot(f1_epochs, f1_vals, 'm-o', linewidth=2, markersize=4)
axes[7].set_title('Validation F1-Score', fontsize=14)
axes[7].set_xlabel('Epoch')
axes[7].set_ylabel('F1-Score')
axes[7].set_ylim(0, 1)
axes[7].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.WORK_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## 14. Đánh giá trên tập test

In [ ]:
# ---- Ham chon nguong / tinh lai metric tai 1 nguong ----
def find_best_threshold(y_true, y_score):
    """Quet nguong 0.05..0.95, tra ve nguong cho F1 cao nhat (dung TAP VALID, khong dung test)."""
    thrs = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y_true, (y_score >= t).astype(int), zero_division=0) for t in thrs]
    k = int(np.argmax(f1s))
    return float(thrs[k]), float(f1s[k])

def metrics_at(y_true, y_score, thr):
    y_pred = (y_score >= thr).astype(int)
    return {
        'y_true': y_true, 'y_score': y_score, 'y_pred': y_pred,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]),
    }

# Load best model
best_model_path = glob.glob(os.path.join(model_dir, 'model_best_*.pth.tar'))
if best_model_path:
    # Neu co nhieu file best, lay file moi nhat (epoch cao nhat)
    best_model_path = sorted(best_model_path, key=os.path.getmtime)[-1]
    print(f'Loading best model: {best_model_path}')
    checkpoint = torch.load(best_model_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'])
    print(f'Loaded model from epoch {checkpoint["epoch"]}')
else:
    print('No best model found, using latest model state.')

# ---- Chon nguong tren tap VALID bang best model ----
print('\n--- Chon nguong tren tap valid ---')
_, _, _, v_det = test_model(valid_loader, model, cuda, return_details=True)
if len(v_det['y_true']) > 0:
    BEST_THR, v_f1 = find_best_threshold(v_det['y_true'], v_det['y_score'])
else:
    BEST_THR, v_f1 = 0.5, 0.0
print(f'Nguong tot nhat tren valid: {BEST_THR:.2f} (valid F1 = {v_f1:.4f}) -> co dinh nguong nay cho test')

# Create test loader
test_cfg = copy.deepcopy(cfg)
test_cfg.USING_PSEUDO_FAKE = False

test_loader = get_rawaudio_data(test_transform, test_cfg, 'test')
print(f'Test loader: {len(test_loader)} samples')

print('\n--- Running Evaluation ---')
auc, real_dist, fake_dist, test_metrics = test_model(
    test_loader, model, cuda, return_details=True
)
print(f'(Tham khao) Test o nguong 0.5: F1={test_metrics["f1"]:.4f}  Acc={test_metrics["accuracy"]:.4f}')

# Tinh lai metric test voi nguong da chon tren valid (AUC khong doi vi khong phu thuoc nguong)
test_metrics = metrics_at(test_metrics['y_true'], test_metrics['y_score'], BEST_THR)

y_true = test_metrics['y_true']
y_score = test_metrics['y_score']
y_pred = test_metrics['y_pred']

print('\n' + '=' * 60)
print(f'  FINAL RESULTS (nguong = {BEST_THR:.2f}, chon tren valid)')
print(f'  AUC Score:  {auc:.4f}')
print(f'  Accuracy:   {test_metrics["accuracy"]:.4f}')
print(f'  Precision:  {test_metrics["precision"]:.4f}')
print(f'  Recall:     {test_metrics["recall"]:.4f}')
print(f'  F1-Score:   {test_metrics["f1"]:.4f}')
print(f'  Real Distance: {real_dist:.4f}')
print(f'  Fake Distance: {fake_dist:.4f}')
print('=' * 60)


In [ ]:
class_names = ['Real', 'Fake']

# ----- Confusion Matrix -----
cm = test_metrics['confusion_matrix']
print('Confusion Matrix (hang = thuc te, cot = du doan):')
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel='Predicted label',
    ylabel='True label',
    title='Confusion Matrix'
)
thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, format(cm[row, col], 'd'),
                ha='center', va='center',
                color='white' if cm[row, col] > thresh else 'black',
                fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(cfg.WORK_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ----- Classification Report -----
print('\nClassification Report:')
report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report_str)

# Luu classification report ra file text
with open(os.path.join(cfg.WORK_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report_str)
print(f"Da luu classification report vao {os.path.join(cfg.WORK_DIR, 'classification_report.txt')}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ----- ROC Curve -----
fpr, tpr, _ = roc_curve(y_true, y_score)
axes[0].plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC curve (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', linewidth=1, linestyle='--', label='Random guess')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=14)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# ----- Precision-Recall Curve -----
prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_score)
axes[1].plot(rec_curve, prec_curve, color='green', linewidth=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.WORK_DIR, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# ----- Bang tong hop chi so cuoi cung -----
summary_df = pd.DataFrame({
    'Metric': ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [auc, test_metrics['accuracy'], test_metrics['precision'],
              test_metrics['recall'], test_metrics['f1']]
})
print('Bang tong hop chi so danh gia tren tap test:')
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(cfg.WORK_DIR, 'test_metrics_summary.csv'), index=False)

## 15. Lưu model cuối

In [ ]:
final_model_path = os.path.join(cfg.WORK_DIR, 'fgi_deepfake_detector_final.pth')
torch.save({
    'state_dict': model.state_dict(),
    'config': {
        'with_attention': cfg.WITH_ATTENTION,
        'residual_conn': cfg.RESIDUAL_CONN,
        'spatial_size': cfg.SPATIAL_SIZE,
        'net': cfg.NET,
    },
    'auc': auc,
}, final_model_path)

file_size = os.path.getsize(final_model_path) / (1024 * 1024)
print(f'Final model saved: {final_model_path} ({file_size:.2f} MB)')
print(f'AUC: {auc:.4f}')

## 16. Suy luận trên video mới

Cần ffmpeg và mediapipe (crop miệng), nên cài lại ở hai cell đầu tiên bên dưới (chỉ cần chạy nếu muốn dự đoán video mới).

In [ ]:
!pip uninstall -y -q imageio-ffmpeg
!pip install -q "imageio-ffmpeg==0.4.5"
!pip install -q mediapipe   # crop mieng (CROP_MODE = 'mouth_lm') can mediapipe, giong luc tien xu ly

In [ ]:
import imageio_ffmpeg, os

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print('ffmpeg binary tại:', ffmpeg_path)

os.makedirs('/usr/local/bin', exist_ok=True)
symlink_path = '/usr/local/bin/ffmpeg'
if os.path.exists(symlink_path) or os.path.islink(symlink_path):
    os.remove(symlink_path)
os.symlink(ffmpeg_path, symlink_path)

import subprocess
result = subprocess.run(['ffmpeg', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print('ffmpeg return code:', result.returncode)
print(result.stdout.decode()[:200])

In [ ]:
# ===== PHAI GIONG HET notebook 1 (cung tham so, ke ca CROP_MODE) de train/suy luan nhat quan =====
import os, shutil, subprocess, wave, threading, urllib.request
from collections import Counter   # khong dung "import collections": notebook 2 da alias ten do sang collections.abc
import numpy as np
import cv2

# ---- tham so (co the chinh) ----
SCAN_SECONDS     = 8      # quet toi da bay nhieu giay dau cua video de tim doan hop le
FRAME_SHORT_SIDE = 540    # canh ngan cua frame trung gian (de crop mat; lon hon = net hon nhung cham hon)
CUT_THRESH       = 0.45   # khoang cach Bhattacharyya giua 2 frame lien tiep > nguong => co chuyen canh (tang nguong = it loai hon)
FACE_SAMPLE_IDX  = (0, 7, 15, 22, 29)   # cac frame (trong 30 frame cua doan) dung de do mat
MIN_FACE_FRAC    = 0.6    # can tim thay mat o it nhat 60% so frame mau
CROP_MODE        = 'mouth_lm'   # 'mouth_lm' = YuNet tim mat + MediaPipe lay landmark MOI -> o crop bam sat mieng that (can mediapipe)
                                # 'mouth'    = o vuong theo kich thuoc mat + 2 mep mieng cua YuNet (khong can mediapipe)
                                # 'face'     = crop ca mat (cach cu)
MIN_FACE_PX      = 64 if CROP_MODE in ('mouth', 'mouth_lm') else 48   # canh mat toi thieu (frame trung gian); o crop mieng nho hon nen can mat lon hon cho du net
MAX_UNION_RATIO  = 2.0    # vung chua tat ca mat mau / kich thuoc mat trung binh; lon hon => mat di chuyen qua nhieu hoac doi nguoi
FACE_MARGIN      = 1.4    # he so mo rong box mat (1.0 = sat mat; 1.4 = lay them tran/cam/tai) - chi dung khi CROP_MODE = 'face'
MOUTH_SCALE      = 0.7    # canh o vuong quanh mieng = MOUTH_SCALE * canh mat (0.7: tu duoi mui den cam; 0.6 = chat hon, 0.8 = rong hon)
MOUTH_Y_SHIFT    = 0.05   # day tam o crop xuong duoi (ti le theo canh o) de co them ham/cam
MOUTH_MARGIN     = 1.1    # he so mo rong sau khi hop nhat cac o mieng trong 1 doan (mat dich chuyen nhe van khong cham mep)
LIP_SCALE         = 1.6   # ('mouth_lm') canh o = LIP_SCALE * be rong moi do tu 40 diem landmark (1.6: tu duoi mui den cam; 1.4 = chat hon, 1.8 = rong hon)
LIP_MIN_FACE_FRAC = 0.5   # ('mouth_lm') o khong nho hon 50% canh mat (tranh qua chat khi chum moi "u")
LANDMARKER_URL    = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
LANDMARKER_PATH   = '/tmp/face_landmarker.task'
OUT_SIZE         = 224
JPEG_QUALITY     = 95

YUNET_URL  = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'
YUNET_PATH = '/tmp/face_detection_yunet_2023mar.onnx'

# ---- thong ke (de biet bao nhieu doan bi loai va vi sao) ----
STATS = Counter()
_stats_lock = threading.Lock()
def _stat(key):
    with _stats_lock:
        STATS[key] += 1

# ---- bo do mat: YuNet (nhanh tren CPU); neu khong tai duoc thi dung Haar cua OpenCV ----
_model_lock = threading.Lock()
_yunet_ok = None
def _yunet_available():
    global _yunet_ok
    with _model_lock:
        if _yunet_ok is not None:
            return _yunet_ok
        try:
            if not (os.path.exists(YUNET_PATH) and os.path.getsize(YUNET_PATH) > 100_000):
                urllib.request.urlretrieve(YUNET_URL, YUNET_PATH)
            ok = os.path.getsize(YUNET_PATH) > 100_000 and hasattr(cv2, 'FaceDetectorYN')
        except Exception as e:
            print('Khong tai duoc YuNet, chuyen sang Haar (kem hon):', e)
            ok = False
        _yunet_ok = ok
        print('Bo do mat:', 'YuNet' if ok else 'Haar cascade')
        return ok

class _FaceDetector:
    def __init__(self):
        self.yunet = None
        self.haar = None
        if _yunet_available():
            self.yunet = cv2.FaceDetectorYN.create(YUNET_PATH, '', (320, 320), 0.7, 0.3, 5000)
        else:
            self.haar = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    def detect(self, bgr):
        """Tra ve list (x, y, w, h, tam_mieng_x, tam_mieng_y) theo toa do cua anh dau vao."""
        h, w = bgr.shape[:2]
        if self.yunet is not None:
            self.yunet.setInputSize((w, h))
            _, faces = self.yunet.detect(bgr)
            if faces is None:
                return []
            # f = [x, y, w, h, mat phai(2), mat trai(2), mui(2), mep mieng phai(2), mep mieng trai(2), score]
            return [(float(f[0]), float(f[1]), float(f[2]), float(f[3]),
                     (float(f[10]) + float(f[12])) / 2, (float(f[11]) + float(f[13])) / 2) for f in faces]
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        found = self.haar.detectMultiScale(gray, 1.1, 5, minSize=(24, 24))
        out = []
        for (x, y, w, h) in found:    # Haar khong co diem moc -> uoc luong tam mieng ~78% chieu cao mat
            out.append((float(x), float(y), float(w), float(h), float(x) + w / 2.0, float(y) + 0.78 * h))
        return out

_tls = threading.local()
def _get_detector():
    if not hasattr(_tls, 'det'):          # moi luong 1 bo do rieng (OpenCV detector khong an toan da luong)
        _tls.det = _FaceDetector()
    return _tls.det

# ---- doc frame / phat hien chuyen canh ----
def _fpath(frames_dir, i):
    return os.path.join(frames_dir, '%06d.jpg' % i)

def _read_small(path):
    return cv2.imread(path, cv2.IMREAD_REDUCED_COLOR_2)   # giai ma nua kich thuoc -> nhanh gap nhieu lan

def _hist(small):
    t = cv2.resize(small, (64, 36), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(t, cv2.COLOR_BGR2HSV)
    h = cv2.calcHist([hsv], [0, 1], None, [16, 8], [0, 180, 0, 256])
    cv2.normalize(h, h, 1, 0, cv2.NORM_L1)
    return h

def has_scene_cut(smalls):
    prev, worst = None, 0.0
    for s in smalls:
        h = _hist(s)
        if prev is not None:
            worst = max(worst, cv2.compareHist(prev, h, cv2.HISTCMP_BHATTACHARYYA))
        prev = h
    return worst > CUT_THRESH, worst

# ---- chon mat + tinh box crop cho ca doan ----
def _pick_face(faces, prev_center):
    if not faces:
        return None
    if prev_center is None:
        return max(faces, key=lambda f: f[2] * f[3])
    return min(faces, key=lambda f: (f[0] + f[2] / 2 - prev_center[0]) ** 2 + (f[1] + f[3] / 2 - prev_center[1]) ** 2)

# ---- MediaPipe Face Landmarker (API Tasks): lay 40 diem landmark MOI trong vung mat YuNet tim duoc. Chi dung khi CROP_MODE = 'mouth_lm' ----
_lm_lock = threading.Lock()
_lm_ready = None
LM_STATS = Counter()      # 'lm_ok' = o crop tu landmark moi; 'lm_fallback' = khong lay duoc -> dung o theo mep mieng YuNet (khong tinh vao STATS)

def _lip_indices():
    from mediapipe.tasks.python import vision
    conns = vision.FaceLandmarksConnections.FACE_LANDMARKS_LIPS
    return sorted({i for c in conns for i in (c.start, c.end)})

def _get_landmarker():
    if not hasattr(_tls, 'lm'):          # moi luong 1 bo rieng (khong dung chung giua cac luong)
        from mediapipe.tasks.python import vision, BaseOptions
        opts = vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=LANDMARKER_PATH),
            running_mode=vision.RunningMode.IMAGE, num_faces=1,
            min_face_detection_confidence=0.3, min_face_presence_confidence=0.3)
        _tls.lm = vision.FaceLandmarker.create_from_options(opts)
        _tls.lip_idx = np.array(_lip_indices())
    return _tls.lm, _tls.lip_idx

def _landmarker_available():
    global _lm_ready
    with _lm_lock:
        if _lm_ready is not None:
            return _lm_ready
        try:
            import mediapipe                      # noqa: F401  (bao loi som neu chua cai)
            if not (os.path.exists(LANDMARKER_PATH) and os.path.getsize(LANDMARKER_PATH) > 1_000_000):
                urllib.request.urlretrieve(LANDMARKER_URL, LANDMARKER_PATH)
            ok = os.path.getsize(LANDMARKER_PATH) > 1_000_000
            if ok:
                _get_landmarker()                 # thu tao that de chac chan file model dung
        except Exception as e:
            print('Khong dung duoc MediaPipe Face Landmarker:', repr(e))
            ok = False
        _lm_ready = ok
        print('Bo landmark moi:', 'MediaPipe Face Landmarker' if ok else 'KHONG CO')
        return ok

def _lip_pts(rgb):
    """Tra ve mang (40, 2) toa do CHUAN HOA (0..1, theo anh dua vao) cua cac diem moi, hoac None."""
    import mediapipe as mp
    lm, idx = _get_landmarker()
    res = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    if not res.face_landmarks:
        return None
    p = res.face_landmarks[0]
    return np.array([(p[i].x, p[i].y) for i in idx], dtype=np.float32)

def _lip_box_from_pts(pts, crop_rect, f):
    """pts: diem moi chuan hoa trong o crop_rect=(x1, y1, cw, ch); f = (x, y, w, h, mx, my) cua mat. Tra ve o vuong (x, y, w, h) quanh MOI hoac None."""
    if pts is None:
        return None
    x1, y1, cw, ch = crop_rect
    px = pts[:, 0] * cw + x1
    py = pts[:, 1] * ch + y1
    lx1, lx2, ly1, ly2 = float(px.min()), float(px.max()), float(py.min()), float(py.max())
    lip_w = lx2 - lx1
    face_side = 0.5 * (f[2] + f[3])
    cx, cy = (lx1 + lx2) / 2, (ly1 + ly2) / 2
    # loai ket qua bat thuong: moi qua be/lon so voi mat, hoac lech xa vi tri mieng ma bo do mat uoc luong (co the bat nham mat khac)
    if not (0.2 * face_side <= lip_w <= 0.8 * face_side):
        return None
    if ((cx - f[4]) ** 2 + (cy - f[5]) ** 2) ** 0.5 > 0.25 * face_side:
        return None
    side = max(LIP_SCALE * lip_w, LIP_MIN_FACE_FRAC * face_side)
    cy = cy + MOUTH_Y_SHIFT * side
    return (cx - side / 2, cy - side / 2, side, side)

def _lip_box(full_bgr, f):
    H, W = full_bgr.shape[:2]
    x, y, w, h = f[:4]
    m = 0.25 * max(w, h)                       # mo rong box mat de landmarker thay du dau/cam
    x1, y1 = int(max(0, x - m)), int(max(0, y - m))
    x2, y2 = int(min(W, x + w + m)), int(min(H, y + h + m))
    if x2 - x1 < 16 or y2 - y1 < 16:
        return None
    pts = _lip_pts(cv2.cvtColor(full_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    return _lip_box_from_pts(pts, (x1, y1, x2 - x1, y2 - y1), f)

def _sample_box(frames_dir, idx, f):
    """O crop cho 1 frame mau (toa do frame trung gian) theo CROP_MODE."""
    if CROP_MODE == 'mouth_lm':
        full = cv2.imread(_fpath(frames_dir, idx))
        b = _lip_box(full, f) if full is not None else None
        with _stats_lock:
            LM_STATS['lm_ok' if b is not None else 'lm_fallback'] += 1
        return b if b is not None else _mouth_box(f)
    if CROP_MODE == 'mouth':
        return _mouth_box(f)
    return f[:4]

if CROP_MODE == 'mouth_lm' and not _landmarker_available():
    raise RuntimeError("CROP_MODE = 'mouth_lm' nhung khong dung duoc MediaPipe Face Landmarker (xem thong bao o tren). "
                       "Cach xu ly: !pip install -q mediapipe (xong khoi dong lai runtime neu Colab yeu cau), kiem tra mang tai duoc "
                       "file model, hoac dat CROP_MODE = 'mouth' (khong can mediapipe). Khong duoc tu doi cach crop giua luc train va suy luan.")

def _mouth_box(f):
    """Tu (x, y, w, h, mouth_cx, mouth_cy) cua 1 khuon mat -> o vuong (x, y, w, h) quanh MIENG.
    Canh o tinh theo kich thuoc mat (on dinh hon do theo 2 mep mieng vi diem moc co the nhieu)."""
    x, y, w, h, mx, my = f
    side = MOUTH_SCALE * 0.5 * (w + h)
    cy = my + MOUTH_Y_SHIFT * side
    return (mx - side / 2, cy - side / 2, side, side)

def analyze_chunk(frames_dir, start):
    """Phan tich doan 30 frame bat dau tu frame so `start` (danh so tu 1).
    Tra ve (box, ly_do). box = (cx, cy, canh) tren frame trung gian, hoac None neu doan bi loai."""
    smalls = [_read_small(_fpath(frames_dir, i)) for i in range(start, start + 30)]
    if any(s is None for s in smalls):
        return None, 'doc_frame_loi'
    cut, _ = has_scene_cut(smalls)
    if cut:
        return None, 'chuyen_canh'
    det = _get_detector()
    boxes, prev = [], None
    for k in FACE_SAMPLE_IDX:
        faces = [tuple(v * 2 for v in f) for f in det.detect(smalls[k])]   # x2: ve toa do frame trung gian
        faces = [f for f in faces if min(f[2], f[3]) >= MIN_FACE_PX]
        f = _pick_face(faces, prev)
        if f is not None:
            boxes.append(_sample_box(frames_dir, start + k, f))
            prev = (f[0] + f[2] / 2, f[1] + f[3] / 2)    # van theo doi theo tam MAT (de khong nhay sang nguoi khac)
    if len(boxes) < MIN_FACE_FRAC * len(FACE_SAMPLE_IDX):
        return None, 'khong_co_mat'
    x1 = min(b[0] for b in boxes); y1 = min(b[1] for b in boxes)
    x2 = max(b[0] + b[2] for b in boxes); y2 = max(b[1] + b[3] for b in boxes)
    mean_side = float(np.mean([max(b[2], b[3]) for b in boxes]))
    extent = max(x2 - x1, y2 - y1)
    if extent > MAX_UNION_RATIO * mean_side:
        return None, 'mat_di_chuyen_nhieu'
    margin = MOUTH_MARGIN if CROP_MODE in ('mouth', 'mouth_lm') else FACE_MARGIN
    return ((x1 + x2) / 2, (y1 + y2) / 2, extent * margin), 'ok'

def crop_chunk(frames_dir, start, box, out_size=OUT_SIZE):
    """Crop cung 1 o vuong cho ca 30 frame (giu nhat quan theo thoi gian). Tra ve list anh BGR out_size x out_size."""
    cx, cy, side = box
    first = cv2.imread(_fpath(frames_dir, start))
    H, W = first.shape[:2]
    s = int(round(min(side, H, W)))
    x1 = int(round(cx - s / 2)); y1 = int(round(cy - s / 2))
    x1 = max(0, min(x1, W - s)); y1 = max(0, min(y1, H - s))
    out = []
    for i in range(start, start + 30):
        img = first if i == start else cv2.imread(_fpath(frames_dir, i))
        out.append(cv2.resize(img[y1:y1 + s, x1:x1 + s], (out_size, out_size), interpolation=cv2.INTER_AREA))
    return out

# ---- ffmpeg: tach frame (giu ti le khung hinh) + audio ----
def extract_frames_audio(video_path, work_dir, seconds=SCAN_SECONDS):
    os.makedirs(work_dir, exist_ok=True)
    audio_path = os.path.join(work_dir, 'audio.wav')
    cap = cv2.VideoCapture(video_path)
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if W > 0 and H > 0 and min(W, H) > FRAME_SHORT_SIDE:
        f = FRAME_SHORT_SIDE / min(W, H)
        W, H = int(round(W * f / 2) * 2), int(round(H * f / 2) * 2)
    vf = f'scale={W}:{H}' if W > 0 and H > 0 else f'scale=-2:{FRAME_SHORT_SIDE}'
    t = ['-t', str(seconds)]
    cmd = (['ffmpeg', '-y', '-v', 'error', '-i', video_path]
           + t + ['-r', '30', '-qscale:v', '2', '-vf', vf, '-threads', '1', '-f', 'image2', os.path.join(work_dir, '%06d.jpg')]
           + t + ['-vn', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '48000', audio_path])
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=300)
    n_frames = len([f for f in os.listdir(work_dir) if f.endswith('.jpg')])
    return (audio_path if os.path.exists(audio_path) else None), n_frames

def select_valid_chunks(frames_dir, n_frames, max_chunks):
    """Duyet tung giay, giu lai toi da max_chunks doan hop le. Tra ve list (sec, box)."""
    valid = []
    for sec in range(n_frames // 30):
        if max_chunks and len(valid) >= max_chunks:
            break
        box, why = analyze_chunk(frames_dir, sec * 30 + 1)
        _stat(why)
        if box is not None:
            valid.append((sec, box))
    return valid


def predict_video(video_path, model, transform, cuda_device, max_chunks=10, scan_seconds=12, verbose=True):
    """
    Du doan deepfake cho 1 video, DUNG CUNG cach tien xu ly voi luc train:
    bo doan co chuyen canh / khong co mat, crop vung MIENG theo CROP_MODE (cung 1 box cho ca doan), resize 224x224.
    Tra ve xac suat FAKE (gan 1 = FAKE, gan 0 = REAL), hoac None neu khong co doan hop le.
    """
    from PIL import Image
    model.eval()
    temp_dir = '/mnt/local-scratch/work/temp_inference'
    if not os.access(os.path.dirname(temp_dir), os.W_OK):
        temp_dir = '/tmp/temp_inference'
    shutil.rmtree(temp_dir, ignore_errors=True)
    os.makedirs(temp_dir, exist_ok=True)
    before = Counter(STATS)

    try:
        audio_path, n_frames = extract_frames_audio(video_path, temp_dir, scan_seconds)
        if audio_path is None:
            if verbose: print('Video khong co audio -> khong du doan duoc.')
            return None
        sample_rate, full_audio = wavfile.read(audio_path)
        valid = select_valid_chunks(temp_dir, n_frames, max_chunks)
        if verbose:
            diff = {k: STATS[k] - before[k] for k in STATS if STATS[k] - before[k] > 0}
            print(f'Doan hop le: {len(valid)} | thong ke: {diff}')
        predictions = []
        with torch.no_grad():
            for sec, box in valid:
                crops = crop_chunk(temp_dir, sec * 30 + 1, box)
                chunk_frames = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops]
                t_seq = transform(chunk_frames)
                (C, H, W) = t_seq[0].size()
                t_seq = torch.stack(t_seq, 0)
                t_seq = t_seq.view(1, 1, 30, C, H, W).transpose(2, 3)

                chunk_audio = full_audio[int(sec * sample_rate):int((sec + 1) * sample_rate)]
                if len(chunk_audio) < 48000:
                    chunk_audio = np.pad(chunk_audio, (0, 48000 - len(chunk_audio)))
                elif len(chunk_audio) > 48000:
                    chunk_audio = chunk_audio[:48000]
                norm_audio = min_max_normalize(chunk_audio, int(chunk_audio.min()), int(chunk_audio.max()))
                norm_audio = torch.from_numpy(norm_audio.astype(float)).float()
                norm_audio = ((norm_audio - 0.5) / 0.5).unsqueeze(0)

                pred, _, _ = model(t_seq.to(cuda_device), norm_audio.to(cuda_device))
                predictions.append(pred[0].item())
        return float(np.mean(predictions)) if predictions else None
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

print(f'Inference function defined (CROP_MODE={CROP_MODE} + scene-cut filter).')
print('Usage: score = predict_video("/path/to/video.mp4", model, test_transform, cuda)')
print('  score close to 1 = FAKE, close to 0 = REAL')


In [ ]:
# Example inference (uncomment to test):

# sample_video = '/content/data/deepfake-detection-challenge/train_sample_videos/aaqaifqrwn.mp4'
# if os.path.exists(sample_video):
#     score = predict_video(sample_video, model, test_transform, cuda)
#     if score is not None:
#         label = 'FAKE' if score > 0.5 else 'REAL'
#         print(f'Video: {os.path.basename(sample_video)}')
#         print(f'Score: {score:.4f} -> Prediction: {label}')

print('\nNotebook complete!')
print('To train on full DFDC dataset, set MAX_VIDEOS = None in the preprocessing cell.')

## 17. Đóng gói kết quả để tải về

In [ ]:
import os, json, shutil, glob
from google.colab import files

OUT = '/content/results'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)

# 1) Luu lich su metric (de sau nay ve lai bieu do)
try:
    history = {
        'train_losses': train_losses,
        'train_real_dists': train_real_dists,
        'train_fake_dists': train_fake_dists,
        'valid_aucs': valid_aucs,
        'valid_accuracies': valid_accuracies,
        'valid_precisions': valid_precisions,
        'valid_recalls': valid_recalls,
        'valid_f1s': valid_f1s,
        'valid_loglosses': valid_loglosses,
    }
    with open(os.path.join(OUT, 'history.json'), 'w') as f:
        json.dump(history, f, default=float)
except NameError as e:
    print('Bo qua history:', e)

# 2) Bieu do, bao cao, model cuoi, file split
names = ['training_curves.png', 'confusion_matrix.png', 'roc_pr_curves.png',
         'classification_report.txt', 'test_metrics_summary.csv',
         'fgi_deepfake_detector_final.pth',
         'valid_metrics_per_epoch.csv',
         'train_split.csv', 'valid_split.csv', 'test_imbalance_split.csv']
for name in names:
    src = os.path.join(cfg.WORK_DIR, name)
    if os.path.exists(src):
        shutil.copy(src, OUT)
        print('Da them:', name)
    else:
        print('Khong thay:', name)

# 3) Checkpoint tot nhat + TensorBoard log
for pattern in ['model_best_*.pth.tar']:
    for p in glob.glob(os.path.join(model_dir, pattern)):
        os.makedirs(os.path.join(OUT, 'models'), exist_ok=True)
        shutil.copy(p, os.path.join(OUT, 'models'))
        print('Da them:', os.path.basename(p))
if os.path.exists(log_dir):
    shutil.copytree(log_dir, os.path.join(OUT, 'logs'), dirs_exist_ok=True)

# 4) Nen zip va tai ve
zip_path = shutil.make_archive('/content/results', 'zip', OUT)
print(f'Zip: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
files.download(zip_path)